In [69]:
# --  import libraries
import pandas as pd
import numpy as np
import pickle
import re
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# In this notebook we will be predicting the IVL using the Models we made

In [70]:
# read in data
matches_reference = pd.read_csv('matches_updated (1).csv'); display(matches_reference); print()
inference_pipeline_db = pd.read_csv('inference_pipeline_db.csv'); display(inference_pipeline_db); print()
teams = pd.read_csv('teams.xls'); display(teams); print()
duals = pd.read_csv('dual_meets.xls'); display(duals); print()
wrestlers = pd.read_csv('wrestlers_updated.csv'); display(wrestlers); print()

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
0,280,141,11/1/2025,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,LDEC,LDEC 3 - 2,JR,Duke,JR,Sacred Heart
1,283,197,11/1/2025,878.0,Kael Bennie,46.0,263.0,Angelo Posada,20.0,False,LDEC,LDEC 5 - 4,SO,Utah Valley,FR,Stanford
2,283,285,11/1/2025,879.0,Jack Forbes,62.0,429.0,Jackson Mankowski,182.0,True,WDEC,WDEC 10 - 3,SR,Utah Valley,SO,Stanford
3,284,125,11/1/2025,784.0,Koda Holeman,40.0,530.0,Jacob Macatangay,69.0,True,WMD,WMD 20 - 8,JR,Cal Poly,JR,Purdue
4,284,133,11/1/2025,785.0,Anthony Lucio,245.0,226.0,Blake Boarman,57.0,False,LDEC,LDEC 8 - 2,RSFR,Cal Poly,SR,Purdue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5835,547,174,2/23/2026,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,SO,Presbyterian,FR,VMI
5836,547,184,2/23/2026,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,JR,Presbyterian,FR,VMI
5837,547,197,2/23/2026,911.0,Toler Hornick,207.0,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,SO,Presbyterian,JR,VMI
5838,547,149,2/23/2026,398.0,Rey Ortiz,245.0,811.0,Patrick Jordon,77.0,False,LFALL,LFALL 4:22,SO,Presbyterian,JR,VMI


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff
0,William Morrow,Bloomsburg,206,75,[157],22,3,19,13.6,1,4.5,1,0,0,2,113.41,2.68,-2.77,5.20
1,Mason Rebuck,Bloomsburg,53,75,[285],20,15,5,75.0,10,50.0,2,2,6,5,127.85,8.17,3.92,8.62
2,Noah Mulvaney,Bucknell,32,36,[165],20,17,3,85.0,7,35.0,3,3,1,10,93.00,9.22,5.33,6.26
3,Jordan Soriano,Drexel,60,44,[141],20,14,6,70.0,7,35.0,0,2,5,7,112.05,7.31,2.23,6.22
4,Jesse Mendez,Ohio State,1,2,[141],19,19,0,100.0,17,89.5,9,3,5,2,57.37,16.64,12.36,4.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,Alaa Aly,Edinboro,208,47,[184],1,0,1,0.0,0,0.0,0,0,0,0,40.00,5.00,-15.00,0.00
1502,Cole Tolley,West Virginia,114,17,[197],1,0,1,0.0,0,0.0,0,0,0,0,63.00,6.00,-15.00,0.00
1503,Winston McBride,Wyoming,182,21,[285],1,1,0,100.0,0,0.0,0,0,0,1,240.00,1.00,1.00,0.00
1504,Aiden Robertson,Air Force,222,52,[174],1,0,1,0.0,0,0.0,0,0,0,0,76.00,0.00,-5.00,0.00


,team_id,name
0,1,Northern Iowa
1,2,South Dakota State
2,3,Drexel
3,4,Northern Illinois
4,5,Central Michigan
...,...,...
73,74,Indiana
74,75,Army West Point
75,76,Wyoming
76,77,Bellarmine


,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
0,1,1,14,20,2,20,14,01/10/26
1,2,1,14,22,3,42,16,01/10/26
2,3,3,42,21,4,45,12,01/10/26
3,4,2,20,28,5,50,10,01/10/26
4,5,5,50,23,6,55,15,01/10/26
...,...,...,...,...,...,...,...,...
579,580,50,26,22,33,35,18,02/19/26
580,581,63,47,28,8,75,10,02/19/26
581,582,44,39,37,34,57,3,02/19/26
582,583,54,42,32,60,65,14,02/19/26


,wrestler_id,name,Class,Team
0,1,Bowen Downey,SO,Northern Iowa
1,2,Julian Farber,SR,Northern Iowa
2,3,Max Brady,FR,Northern Iowa
3,4,Ethan Basile,SR,Northern Iowa
4,5,Cael Rahnavardi,SR,Northern Iowa
...,...,...,...,...
1507,38,Landen Johnson,SO,Northern Illinois
1508,168,Nathan Taylor,SR,Lehigh
1509,897,James Conway,SR,Missouri
1510,2000,James Conway,SR,Franklin & Marshall


In [71]:
# -- Join class to inference_pipeline_db
inference_pipeline_db = inference_pipeline_db.merge(wrestlers[["name", "Team", "Class"]], left_on=['wrestler_name', 'team_name'], right_on = ['name', 'Team'])
inference_pipeline_db = inference_pipeline_db.drop(columns=['name', 'Team'])
inference_pipeline_db

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
0,William Morrow,Bloomsburg,206,75,[157],22,3,19,13.6,1,4.5,1,0,0,2,113.41,2.68,-2.77,5.20,SR
1,Mason Rebuck,Bloomsburg,53,75,[285],20,15,5,75.0,10,50.0,2,2,6,5,127.85,8.17,3.92,8.62,SO
2,Noah Mulvaney,Bucknell,32,36,[165],20,17,3,85.0,7,35.0,3,3,1,10,93.00,9.22,5.33,6.26,JR
3,Jordan Soriano,Drexel,60,44,[141],20,14,6,70.0,7,35.0,0,2,5,7,112.05,7.31,2.23,6.22,SR
4,Jesse Mendez,Ohio State,1,2,[141],19,19,0,100.0,17,89.5,9,3,5,2,57.37,16.64,12.36,4.25,SR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,Alaa Aly,Edinboro,208,47,[184],1,0,1,0.0,0,0.0,0,0,0,0,40.00,5.00,-15.00,0.00,RSFR
1502,Cole Tolley,West Virginia,114,17,[197],1,0,1,0.0,0,0.0,0,0,0,0,63.00,6.00,-15.00,0.00,SO
1503,Winston McBride,Wyoming,182,21,[285],1,1,0,100.0,0,0.0,0,0,0,1,240.00,1.00,1.00,0.00,SO
1504,Aiden Robertson,Air Force,222,52,[174],1,0,1,0.0,0,0.0,0,0,0,0,76.00,0.00,-5.00,0.00,FR


In [72]:
def filter_weights(wrestlers: list, match_details=False):
  if match_details:
    print("Matches Detailed")
    display(matches_reference.query('home_name in @wrestlers or away_name in @wrestlers')); print()

  print("Overall Stats")
  display(inference_pipeline_db.query('wrestler_name in @wrestlers')); print()

wrestlers_125 = ["Davis Motyka", "Greg Diakomihalis", "Logan Brzozowski", "Sulayman Bah", "Chase Yasutake", "Marc-Anthony McGowan"]
filter_weights(wrestlers_125)

Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
284,Sulayman Bah,Columbia,23,23,[125],13,9,4,69.2,6,46.2,4,1,1,3,86.69,11.45,6.00,8.66,JR
405,Logan Brzozowski,Harvard,45,51,[125],12,8,4,66.7,4,33.3,2,1,1,4,84.75,8.18,2.82,8.86,SO
480,Chase Yasutake,Brown,230,62,[125],11,1,10,9.1,1,9.1,0,0,1,0,105.91,0.67,-15.00,0.00,FR
563,Marc-Anthony McGowan,Princeton,13,32,[125],10,8,2,80.0,5,50.0,2,3,0,3,59.30,9.40,7.10,6.14,SO
584,Greg Diakomihalis,Cornell,50,13,[125],9,3,6,33.3,2,22.2,1,1,0,1,66.56,5.38,1.00,7.37,SR
901,Davis Motyka,Pennsylvania,37,26,[125],5,4,1,80.0,1,20.0,1,0,0,3,95.60,7.40,4.80,6.72,FR


In [73]:
# -- Create for all weights
wrestlers_133 = ["Evan Mougalian", "Ethan Rivera", "Coleman Nogle", "Evin Gursoy", "Douglas Shipers", "Tyler Ferrera"]
print("133 IVL WRESTLERS")
filter_weights(wrestlers_133); print(); print()

wrestlers_141 = ["Vince Cornella", "Matthew Martino", "Lorenzo Frezza", "Khimari Manns", "Dante Frinzi", "CJ Composto"]
print("141 IVL WRESTLERS")
filter_weights(wrestlers_141); print(); print()

wrestlers_149 = ["Jaxon Joy", "Richard Fedalen", "Jack Crook", "Eligh Rivera", "Austin McBurney", "Cross Wasilewski"]
print("149 IVL WRESTLERS")
filter_weights(wrestlers_149); print(); print()

wrestlers_157 = ["Meyer Shapiro", "Ethan Mojena", "Kai Owen", "Jimmy Harrington", "Rocco Camillaci", "Jude Swisher"]
print("157 IVL WRESTLERS")
filter_weights(wrestlers_157); print(); print()

wrestlers_165 = ["Cesar Alvan", "Joseph Cangro", "Maximus Norman", "Sean Seefeldt", "Benny Rogers", "Ty Whalen"]
print("165 IVL WRESTLERS")
filter_weights(wrestlers_165); print(); print()

wrestlers_174 = ["Simon Ruiz", "Haden Bottiglieri", "Caden Bellis", "Holden Garcia", "Drew Clearie", "Nick Fine"]
print("174 IVL WRESTLERS")
filter_weights(wrestlers_174); print(); print()

wrestlers_184 = ["Joe Curtis", "Xavier Giles", "Matthew Walsh", "Liam Carlin", "Connor O'Donnell", "Louis Cerchio"]
print("184 IVL WRESTLERS")
filter_weights(wrestlers_184); print(); print()

wrestlers_197 = ["Andrew Reall", "Aiden Hanning", "Hudson Skove", "Martin Cosgrove", "Conor McCloskey", "Jack Wehmeyer"]
print("197 IVL WRESTLERS")
filter_weights(wrestlers_197); print(); print()


wrestlers_285 = ["Vincent Mueller", "Sebastian Garibaldi", "Ashton Davis", "John Pardo", "Daniel Bittner", "Alex Semenenko"]
print("285 IVL WRESTLERS")
filter_weights(wrestlers_285); print(); print()

#all_wrestler_list = [wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149, wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184, wrestlers_197, wrestlers_285]
#weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

133 IVL WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
209,Douglas Shipers,Brown,254,62,[133],14,0,14,0.0,0,0.0,0,0,0,0,93.43,1.50,-11.00,5.66,FR
249,Evin Gursoy,Columbia,68,23,[133],14,10,4,71.4,5,35.7,1,3,1,5,106.21,5.77,3.54,5.95,FR
335,Coleman Nogle,Harvard,72,51,[133],12,6,6,50.0,3,25.0,0,1,2,3,89.33,4.80,-2.60,7.11,JR
462,Tyler Ferrera,Cornell,18,13,[133],11,8,3,72.7,2,18.2,1,1,0,6,68.18,7.36,2.45,5.96,SO
539,Evan Mougalian,Pennsylvania,16,26,[133],10,8,2,80.0,4,40.0,1,1,2,4,90.40,8.38,4.88,5.54,JR
560,Ethan Rivera,Princeton,45,32,[133],10,5,5,50.0,1,10.0,0,0,1,4,56.00,3.56,-1.56,3.40,SO





141 IVL WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
257,Vince Cornella,Cornell,6,13,[141],13,12,1,92.3,5,38.5,1,2,2,7,76.69,7.73,5.27,4.27,RSJR
258,Matthew Martino,Princeton,28,32,"[141, 149]",13,5,8,38.5,2,15.4,0,1,1,3,44.38,6.42,-3.92,7.89,FR
273,Lorenzo Frezza,Columbia,46,23,[141],13,8,5,61.5,5,38.5,3,1,1,3,72.85,10.83,3.17,8.36,JR
495,Khimari Manns,Brown,67,62,[141],11,6,5,54.5,2,18.2,1,1,0,4,71.18,7.80,0.70,9.17,FR
603,Dante Frinzi,Harvard,133,51,[141],9,2,7,22.2,1,11.1,0,0,1,1,77.89,3.83,-5.33,6.53,SR
682,CJ Composto,Pennsylvania,10,26,[141],8,6,2,75.0,3,37.5,1,2,0,3,49.50,10.14,6.14,6.64,SR





149 IVL WRESTLERS
Overall Stats


...

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
138,Louis Cerchio,Cornell,50,13,"[165, 184]",15,6,9,40.0,2,13.3,0,2,0,4,50.00,5.57,0.07,7.04,FR
237,Xavier Giles,Princeton,73,32,"[174, 184]",14,5,9,35.7,2,14.3,0,2,0,3,64.86,5.08,-3.77,7.96,SO
411,Matthew Walsh,Harvard,81,51,[184],12,6,6,50.0,2,16.7,0,1,1,4,99.08,4.70,0.30,4.47,JR
582,Connor O'Donnell,Brown,155,62,[184],9,1,8,11.1,0,0.0,0,0,0,1,67.78,3.67,-7.56,6.41,SO
753,Liam Carlin,Pennsylvania,58,26,[184],7,3,4,42.9,0,0.0,0,0,0,3,82.29,3.71,-1.57,6.75,FR
903,Joe Curtis,Columbia,19,23,[184],5,5,0,100.0,5,100.0,0,1,4,0,99.20,16.00,12.00,0.00,JR





197 IVL WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
285,Conor McCloskey,Princeton,148,32,"[197, 285]",13,0,13,0.0,0,0.0,0,0,0,0,55.62,2.27,-9.09,5.49,FR
384,Andrew Reall,Brown,18,62,[197],12,11,1,91.7,7,58.3,3,3,1,4,86.33,11.18,7.91,6.02,SO
465,Jack Wehmeyer,Columbia,42,23,[197],11,8,3,72.7,5,45.5,2,1,2,3,107.27,7.78,5.56,6.84,JR
482,Martin Cosgrove,Pennsylvania,80,26,[197],11,4,7,36.4,1,9.1,0,1,0,3,74.73,4.73,-0.73,7.00,JR
559,Hudson Skove,Harvard,117,51,"[197, 285]",10,3,7,30.0,0,0.0,0,0,0,3,62.70,3.44,-6.44,8.57,SO
671,Aiden Hanning,Cornell,87,13,[197],8,1,7,12.5,1,12.5,0,0,1,0,48.25,1.14,-4.00,1.91,JR





285 IVL WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
294,Alex Semenenko,Brown,22,62,[285],13,11,2,84.6,6,46.2,2,0,4,5,113.31,7.89,3.22,8.93,SR
374,Vincent Mueller,Columbia,26,23,[285],12,10,2,83.3,7,58.3,1,2,4,3,88.75,7.25,2.75,10.10,JR
531,John Pardo,Pennsylvania,85,26,[285],10,6,4,60.0,3,30.0,2,0,1,3,120.80,7.22,0.22,11.37,SO
668,Sebastian Garibaldi,Princeton,121,32,[285],8,1,7,12.5,1,12.5,0,0,1,0,65.88,2.50,-7.33,6.02,SR
767,Ashton Davis,Cornell,48,13,[285],7,0,7,0.0,0,0.0,0,0,0,0,21.57,2.14,-4.43,3.41,JR
1077,Daniel Bittner,Harvard,174,51,[285],3,1,2,33.3,0,0.0,0,0,0,1,75.67,3.33,-8.00,9.64,SR


# Find Test Cases

In [74]:
matches_reference.query('home_name == "Simon Ruiz" or away_name == "Simon Ruiz"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
490,222,174,11/15/2025,596.0,Alex Facundo,14.0,175.0,Simon Ruiz,5.0,False,LDEC,LDEC 2 - 0,JR,Oklahoma State,SO,Cornell
494,215,174,11/15/2025,33.0,Jared Simma,20.0,175.0,Simon Ruiz,5.0,False,LMD,LMD 18 - 4,SR,Northern Iowa,SO,Cornell
2726,286,174,1/10/2026,503.0,Myles Takats,17.0,175.0,Simon Ruiz,5.0,False,LDEC,LDEC 5 - 2,JR,Bucknell,SO,Cornell
2844,16,174,1/10/2026,165.0,Richie Grungo,106.0,175.0,Simon Ruiz,5.0,False,LTF,LTF5 15 - 0 7:00,SO,Lehigh,SO,Cornell
3074,325,174,1/16/2026,175.0,Simon Ruiz,4.0,964.0,Cooper Haase,68.0,True,WTF,WTF5 20 - 4 5:25,SO,Cornell,SO,Army West Point
3751,360,174,1/24/2026,175.0,Simon Ruiz,4.0,1060.0,Drew Clearie,109.0,True,WFALL,WFALL 4:16,SO,Cornell,SR,Brown
3796,359,174,1/24/2026,175.0,Simon Ruiz,4.0,74.0,Ben Smith,187.0,True,WTF,WTF5 21 - 3 2:39,SO,Cornell,FR,Harvard
4218,396,174,1/31/2026,175.0,Simon Ruiz,4.0,320.0,Cael Valencia,76.0,True,WDEC,WDEC 7 - 5,SO,Cornell,SR,Arizona State
4227,397,174,1/31/2026,175.0,Simon Ruiz,4.0,519.0,Nick Fine,30.0,True,WDEC,WDEC 8 - 5,SO,Cornell,SR,Columbia
4747,507,174,2/7/2026,458.0,Holden Garcia,35.0,175.0,Simon Ruiz,2.0,False,LDEC,LDEC 5 - 1,RSSO,Princeton,SO,Cornell


In [75]:
display(inference_pipeline_db.query('wrestler_name == "Simon Ruiz"'))
display(inference_pipeline_db.query('wrestler_name == "Nick Fine"'))

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
205,Simon Ruiz,Cornell,2,13,[174],14,14,0,100.0,6,42.9,3,2,1,8,55.14,9.85,7.15,5.67,SO


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
305,Nick Fine,Columbia,22,23,[174],13,10,3,76.9,8,61.5,3,4,1,2,81.92,10.0,6.0,7.54,SR


In [76]:
matches_reference.query('home_name == "Jaxon Joy" or away_name == "Jaxon Joy"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
487,222,149,11/15/2025,593.0,Casey Swiderski,21.0,172.0,Jaxon Joy,3.0,False,LDEC,LDEC 4 - 3,RSJR,Oklahoma State,FR,Cornell
565,215,149,11/15/2025,91.0,Caleb Rathjen,44.0,172.0,Jaxon Joy,3.0,False,LMD,LMD 11 - 2,SR,Northern Iowa,FR,Cornell
2022,65,149,12/21/2025,172.0,Jaxon Joy,3.0,609.0,Carter McCallister,138.0,True,WMD,WMD 14 - 5,FR,Cornell,SO,Little Rock
2087,56,149,12/21/2025,172.0,Jaxon Joy,3.0,574.0,William Baysingar,94.0,True,WTF,WTF5 17 - 2 5:37,FR,Cornell,SO,Illinois
2723,286,149,1/10/2026,500.0,Riley Bower,55.0,172.0,Jaxon Joy,3.0,False,LTF,LTF5 17 - 2 7:00,SR,Bucknell,FR,Cornell
2847,16,149,1/10/2026,162.0,Owen Reinsel,48.0,172.0,Jaxon Joy,3.0,False,LMD,LMD 9 - 0,JR,Lehigh,FR,Cornell
3077,325,149,1/16/2026,172.0,Jaxon Joy,3.0,1247.0,Ryan Franco,184.0,True,WTF,WTF5 17 - 1 5:14,FR,Cornell,SR,Army West Point
3748,360,149,1/24/2026,172.0,Jaxon Joy,3.0,1057.0,Austin McBurney,72.0,True,WTF,WTF5 17 - 1 4:37,FR,Cornell,JR,Brown
3793,359,149,1/24/2026,172.0,Jaxon Joy,3.0,71.0,Jack Crook,58.0,True,WMD,WMD 16 - 2,FR,Cornell,JR,Harvard
4211,397,149,1/31/2026,172.0,Jaxon Joy,3.0,516.0,Richard Fedalen,68.0,True,WTF,WTF5 15 - 0 2:29,FR,Cornell,SR,Columbia


In [77]:
display(inference_pipeline_db.query('wrestler_name == "Jaxon Joy"'))
display(inference_pipeline_db.query('wrestler_name == "Jack Crook"'))

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
137,Jaxon Joy,Cornell,3,13,[149],15,14,1,93.3,12,80.0,6,5,1,2,72.67,12.57,10.43,5.96,FR


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
406,Jack Crook,Harvard,60,51,[149],12,8,4,66.7,4,33.3,3,1,0,4,100.08,11.33,3.67,9.39,JR


In [78]:
wrestlers

,wrestler_id,name,Class,Team
0,1,Bowen Downey,SO,Northern Iowa
1,2,Julian Farber,SR,Northern Iowa
2,3,Max Brady,FR,Northern Iowa
3,4,Ethan Basile,SR,Northern Iowa
4,5,Cael Rahnavardi,SR,Northern Iowa
...,...,...,...,...
1507,38,Landen Johnson,SO,Northern Illinois
1508,168,Nathan Taylor,SR,Lehigh
1509,897,James Conway,SR,Missouri
1510,2000,James Conway,SR,Franklin & Marshall


In [79]:
# -- 1 -- BASIC LOGREG (win_rate difference)
with open("logreg_model_OFFICIAL.pkl", "rb") as f:
    logreg_model = pickle.load(f)

# -- 2 -- DECISION TREE
with open("dt_model_tuned_OFFICIAL.pkl", "rb") as f:
    dt_model = pickle.load(f)

# -- 3 -- XGBoost w/o ranks
with open("xgboost_model_tuned_noranks_OFFICIAL.pkl", "rb") as f:
    xgb_model1 = pickle.load(f)

# -- 4 -- XGbosst w rank
with open("xgb_with_ranks_tuned_OFFICIAL.pkl", "rb") as f:
    xgb_model2 = pickle.load(f)



# -- Features Dt and XGBoost(1)
with open("features_dt_xgb1.pkl", "rb") as f:
    features_dt_xgb1 = pickle.load(f)


# -- Features XGBoost(2)
with open("xgb_w_ranks_features.pkl", "rb") as f:
    features_xgb2 = pickle.load(f)

In [80]:
# ============================================
# FUNCTION TO GET MATCHUP HISTORY
# ============================================

def get_wrestler_match_history(wrestler1_name, wrestler2_name, matches_df):
    """
    Get complete match history between two wrestlers from matches_reference.
    Returns dictionary of historical features.
    """

    # Find all matches between these two wrestlers
    mask = (
        ((matches_df['home_name'] == wrestler1_name) & (matches_df['away_name'] == wrestler2_name)) |
        ((matches_df['home_name'] == wrestler2_name) & (matches_df['away_name'] == wrestler1_name))
    )

    history = matches_df[mask].copy().sort_values('event_date')

    if len(history) == 0:
        return {
            'wrestled_before': 0,
            'home_point_diff_rematches': 0,
            'home_pinned_away': 0,
            'total_matches': 0,
            'w1_wins': 0,
            'w2_wins': 0,
            'avg_point_diff_w1': 0,
            'last_match_date': None,
            'last_winner': None
        }

    # Calculate head-to-head stats
    w1_name = wrestler1_name
    w2_name = wrestler2_name

    w1_wins = 0
    w2_wins = 0
    total_point_diff_w1 = 0
    point_diff_count = 0
    pins_by_w1 = 0
    pins_by_w2 = 0

    # Store match results
    match_records = []

    for idx, row in history.iterrows():
        # Determine winner
        if row['home_name'] == w1_name:
            winner = w1_name if row['home_win'] else w2_name
        else:
            winner = w2_name if row['home_win'] else w1_name

        # Calculate point differential (from w1's perspective)
        point_diff = 0
        if 'FALL' not in str(row['win_type']):
            try:
                parts = str(row['Result']).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    score1, score2 = scores[0], scores[1]

                    # Determine scores from w1's perspective
                    if row['home_name'] == w1_name:
                        if row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1
                    else:
                        if not row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1

                    total_point_diff_w1 += point_diff
                    point_diff_count += 1
            except:
                point_diff = 0

        # Check for pins
        was_pin = 'FALL' in str(row['win_type'])
        if was_pin:
            if winner == w1_name:
                pins_by_w1 += 1
            else:
                pins_by_w2 += 1

        # Count wins
        if winner == w1_name:
            w1_wins += 1
        else:
            w2_wins += 1

        # Store match record
        match_records.append({
            'date': row['event_date'],
            'home': row['home_name'],
            'away': row['away_name'],
            'winner': winner,
            'point_diff': point_diff,
            'was_pin': was_pin,
            'result': row['Result'],
            'win_type': row['win_type']
        })

    # Calculate averages
    avg_point_diff_w1 = total_point_diff_w1 / point_diff_count if point_diff_count > 0 else 0

    # Add winner column to history
    history['winner'] = [m['winner'] for m in match_records]

    return {
        'wrestled_before': 1 if len(history) > 0 else 0,
        'home_point_diff_rematches': round(avg_point_diff_w1, 3),
        'home_pinned_away': 1 if pins_by_w1 > 0 else 0,
        'total_matches': len(history),
        'w1_wins': w1_wins,
        'w2_wins': w2_wins,
        'avg_point_diff_w1': round(avg_point_diff_w1, 3),
        'pins_by_w1': pins_by_w1,
        'pins_by_w2': pins_by_w2,
        'last_match_date': history.iloc[-1]['event_date'] if len(history) > 0 else None,
        'last_winner': history.iloc[-1]['winner'] if len(history) > 0 else None,
        'match_history': history
    }


# ============================================
# FUNCTION TO PREPARE FEATURES FOR A MATCHUP
# ============================================

def prepare_matchup_features(w1_name, w2_name, historical_features, inference_db, team_name = "Columbia", weight_class=125):
    """
    Prepare all features needed for models given two wrestlers.
    Columbia always home, otherwise lower rank is home.
    """

    # Get wrestler data from inference pipeline
    w1_data = inference_db[inference_db['wrestler_name'] == w1_name].iloc[0]
    w2_data = inference_db[inference_db['wrestler_name'] == w2_name].iloc[0]

    # Determine home/away: Columbia always home, otherwise by rank (lower rank = better)
    if w1_data['team_name'] == team_name:
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
        # Historical features are from w1's perspective, and home is w1, so no adjustment
        point_diff_adjustment = 1
    elif w2_data['team_name'] == team_name:
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name
        # Historical features are from w1's perspective, but home is w2, so need to flip sign
        point_diff_adjustment = -1
    elif w1_data['rank'] < w2_data['rank']:
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
        point_diff_adjustment = 1
    else:
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name
        point_diff_adjustment = -1

    print(f"\n📌 Home wrestler: {home_name} (Rank: {home['rank']}, Team: {home['team_name']})")
    print(f"📌 Away wrestler: {away_name} (Rank: {away['rank']}, Team: {away['team_name']})")

    # Calculate differential features
    features = {}

    # Basic differentials
    features['matches_wrestled_diff'] = home['total_matches'] - away['total_matches']
    features['win_rate_diff'] = round(home['win_rate']/100 - away['win_rate']/100, 4)
    features['bonus_rate_diff'] = round(home['bonus_rate']/100 - away['bonus_rate']/100, 4)
    features['pin_count_diff'] = home['fall_wins'] - away['fall_wins']
    features['avg_point_diff_diff'] = round(home['avg_point_diff'] - away['avg_point_diff'], 4)
    features['avg_points_scored_diff'] = round(home['avg_points_scored'] - away['avg_points_scored'], 4)
    features['std_point_diff_diff'] = round(home['std_point_diff'] - away['std_point_diff'], 4)
    features['team_rank_diff'] = home['team_rank'] - away['team_rank']

    # Historical features - ADJUST SIGN BASED ON WHO IS HOME
    features['wrestled_before'] = historical_features['wrestled_before']
    # Flip the sign if home is not the same as w1 (the perspective used in historical_features)
    features['home_point_diff_rematches'] = round(historical_features['home_point_diff_rematches'] * point_diff_adjustment, 3)
    features['home_pinned_away'] = historical_features['home_pinned_away'] if point_diff_adjustment == 1 else 0

    # Weight class dummies
    weight_classes = [133, 141, 149, 157, 165, 174, 184, 197, 285]
    for wc in weight_classes:
        features[f'weight_class_{wc}'] = 1 if weight_class == wc else 0

    # Class dummies
    all_classes = ['JR', 'RSFR', 'RSJR', 'RSSO', 'RSSR', 'SO', 'SR']

    for cls in all_classes:
        features[f'home_class_{cls}'] = 1 if home['Class'] == cls else 0

    for cls in all_classes:
        features[f'away_class_{cls}'] = 1 if away['Class'] == cls else 0

    return features, home_name, away_name, home, away


# ============================================


# ============================================
# FIXED FUNCTION TO MAKE PREDICTIONS
# ============================================

def predict_matchup(w1_name, w2_name, features, home_name, away_name):
    """
    Make predictions using all 4 models.
    """

    # Create dataframes with correct feature order
    df_dt = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb1}])
    df_xgb2 = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb2}])

    # Make predictions
    results = {
        'home_wrestler': home_name,
        'away_wrestler': away_name
    }

    # LOGREG
    logreg_input = [[features['win_rate_diff']]]
    logreg_proba = logreg_model.predict_proba(logreg_input)[0]
    logreg_pred = logreg_model.predict(logreg_input)[0]

    results['logreg_pred'] = logreg_pred
    results['logreg_winner'] = home_name if logreg_pred == 1 else away_name
    results['logreg_prob'] = round(logreg_proba[1] if logreg_pred == 1 else logreg_proba[0], 4)

    # DT
    dt_input = df_dt[features_dt_xgb1]
    dt_proba = dt_model.predict_proba(dt_input)[0]
    dt_pred = dt_model.predict(dt_input)[0]

    results['dt_pred'] = dt_pred
    results['dt_winner'] = home_name if dt_pred == 1 else away_name
    results['dt_prob'] = round(dt_proba[1] if dt_pred == 1 else dt_proba[0], 4)

    # XGB1 (no ranks)
    xgb1_input = df_dt[features_dt_xgb1]
    xgb1_proba = xgb_model1.predict_proba(xgb1_input)[0]
    xgb1_pred = xgb_model1.predict(xgb1_input)[0]

    results['xgb1_pred'] = xgb1_pred
    results['xgb1_winner'] = home_name if xgb1_pred == 1 else away_name
    results['xgb1_prob'] = round(xgb1_proba[1] if xgb1_pred == 1 else xgb1_proba[0], 4)

    # XGB2 (with ranks)
    xgb2_input = df_xgb2[features_xgb2]
    xgb2_proba = xgb_model2.predict_proba(xgb2_input)[0]
    xgb2_pred = xgb_model2.predict(xgb2_input)[0]

    results['xgb2_pred'] = xgb2_pred
    results['xgb2_winner'] = home_name if xgb2_pred == 1 else away_name
    results['xgb2_prob'] = round(xgb2_proba[1] if xgb2_pred == 1 else xgb2_proba[0], 4)

    return results


# ============================================
# FUNCTION TO PRINT DETAILED FEATURE INFO
# ============================================

# ============================================
# FUNCTION TO PRINT DETAILED FEATURE INFO
# ============================================

def print_matchup_details(w1_name, w2_name, historical_features, features, results, home_name, away_name, home_data, away_data):
    """
    Print detailed information about a matchup including features and predictions.
    """

    print("\n" + "="*80)
    print(f"🔍 MATCHUP DETAILS: {w1_name} vs {w2_name}")
    print("="*80)

    # Historical Summary
    print(f"\n📜 HISTORICAL SUMMARY:")
    if historical_features['wrestled_before']:
        print(f"   Previous matches: {historical_features['total_matches']}")
        print(f"   {w1_name} wins: {historical_features['w1_wins']}")
        print(f"   {w2_name} wins: {historical_features['w2_wins']}")
        print(f"   Avg point diff ({w1_name}'s perspective): {historical_features['avg_point_diff_w1']}")
        print(f"   Last match: {historical_features['last_match_date']} (Winner: {historical_features['last_winner']})")
    else:
        print("   No previous matches")

    # Historical features for models
    print(f"\n🔍 HISTORICAL FEATURES FOR MODELS:")
    print(f"   wrestled_before: {features['wrestled_before']}")
    print(f"   home_point_diff_rematches: {features['home_point_diff_rematches']}")
    print(f"   home_pinned_away: {features['home_pinned_away']}")

    # Key differentials
    print(f"\n📊 KEY DIFFERENTIALS (home - away):")
    print(f"   win_rate_diff: {features['win_rate_diff']:.4f} ({home_name} {home_data['win_rate']}% - {away_name} {away_data['win_rate']}%)")
    print(f"   matches_wrestled_diff: {features['matches_wrestled_diff']}")
    print(f"   bonus_rate_diff: {features['bonus_rate_diff']:.4f}")
    print(f"   avg_point_diff_diff: {features['avg_point_diff_diff']:.4f}")
    print(f"   team_rank_diff: {features['team_rank_diff']}")

    # Create dataframes to show full feature arrays
    print(f"\n📊 FULL FEATURE ARRAYS:")

    # DT/XGB1 features
    print(f"\n📊 DT/XGB1 features ({len(features_dt_xgb1)} total):")
    df_dt = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb1}])
    for i, feat in enumerate(features_dt_xgb1[:10]):  # Show first 10
        print(f"   {i+1:2d}. {feat:35} = {df_dt[feat].iloc[0]:8.4f}")
    print("   ...")

    # XGB2 features
    print(f"\n📊 XGB2 features ({len(features_xgb2)} total):")
    df_xgb2 = pd.DataFrame([{f: features.get(f, 0) for f in features_xgb2}])
    for i, feat in enumerate(features_xgb2[:10]):  # Show first 10
        print(f"   {i+1:2d}. {feat:35} = {df_xgb2[feat].iloc[0]:8.4f}")
    print("   ...")

    # Predictions
    print(f"\n🔮 PREDICTIONS:")
    print(f"   LOGREG: {results['logreg_winner']} wins with {results['logreg_prob']*100:.1f}% confidence")
    print(f"   DT    : {results['dt_winner']} wins with {results['dt_prob']*100:.1f}% confidence")
    print(f"   XGB1  : {results['xgb1_winner']} wins with {results['xgb1_prob']*100:.1f}% confidence")
    print(f"   XGB2  : {results['xgb2_winner']} wins with {results['xgb2_prob']*100:.1f}% confidence")

    # Check agreement
    winners = [results['logreg_winner'], results['dt_winner'], results['xgb1_winner'], results['xgb2_winner']]
    if all(w == winners[0] for w in winners):
        print(f"\n✅ All models agree: {winners[0]} wins!")
    else:
        print(f"\n⚠️ Models disagree on winner")

# ============================================
# TEST CASE 1: SIMON RUIZ vs NICK FINE
# ============================================

print("\n" + "="*100)
print("🔍 IVY LEAGUE TEST CASE 1: SIMON RUIZ vs NICK FINE")
print("="*100)

# Get their match history
hist_features_ruiz = get_wrestler_match_history(
    "Simon Ruiz", "Nick Fine", matches_reference
)

print(f"\n📜 Raw Historical Features (from Simon's perspective):")
print(f"   wrestled_before: {hist_features_ruiz['wrestled_before']}")
print(f"   home_point_diff_rematches: {hist_features_ruiz['home_point_diff_rematches']}")
print(f"   total_matches: {hist_features_ruiz['total_matches']}")
print(f"   Simon Ruiz wins: {hist_features_ruiz['w1_wins']}")
print(f"   Nick Fine wins: {hist_features_ruiz['w2_wins']}")
print(f"   Avg point diff (Simon's perspective): {hist_features_ruiz['avg_point_diff_w1']}")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
simon_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Simon Ruiz'].iloc[0]
nick_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Nick Fine'].iloc[0]

print(f"   Simon Ruiz - Rank: {simon_data['rank']}, Team: {simon_data['team_name']}, Win Rate: {simon_data['win_rate']}%")
print(f"   Nick Fine - Rank: {nick_data['rank']}, Team: {nick_data['team_name']}, Win Rate: {nick_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff = simon_data['win_rate']/100 - nick_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff:.4f} (Simon {simon_data['win_rate']}% - Nick {nick_data['win_rate']}%)")

# Prepare features for prediction
features_ruiz, home_name_ruiz, away_name_ruiz, home_data_ruiz, away_data_ruiz = prepare_matchup_features(
    "Simon Ruiz", "Nick Fine", hist_features_ruiz, inference_pipeline_db, weight_class=174
)

# Make predictions
results_ruiz = predict_matchup("Simon Ruiz", "Nick Fine", features_ruiz, home_name_ruiz, away_name_ruiz)

# Print detailed information
print_matchup_details(
    "Simon Ruiz", "Nick Fine",
    hist_features_ruiz, features_ruiz, results_ruiz,
    home_name_ruiz, away_name_ruiz, home_data_ruiz, away_data_ruiz
)


# ============================================
# TEST CASE 2: JAXON JOY vs JACK CROOK
# ============================================

print("\n" + "="*100)
print("🔍 IVY LEAGUE TEST CASE 2: JAXON JOY vs JACK CROOK")
print("="*100)

# Get their match history
hist_features_joy = get_wrestler_match_history(
    "Jaxon Joy", "Jack Crook", matches_reference
)

print(f"\n📜 Raw Historical Features (from Jaxon's perspective):")
print(f"   wrestled_before: {hist_features_joy['wrestled_before']}")
print(f"   home_point_diff_rematches: {hist_features_joy['home_point_diff_rematches']}")
print(f"   total_matches: {hist_features_joy['total_matches']}")
print(f"   Jaxon Joy wins: {hist_features_joy['w1_wins']}")
print(f"   Jack Crook wins: {hist_features_joy['w2_wins']}")
print(f"   Avg point diff (Jaxon's perspective): {hist_features_joy['avg_point_diff_w1']}")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
jaxon_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Jaxon Joy'].iloc[0]
jack_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Jack Crook'].iloc[0]

print(f"   Jaxon Joy - Rank: {jaxon_data['rank']}, Team: {jaxon_data['team_name']}, Win Rate: {jaxon_data['win_rate']}%")
print(f"   Jack Crook - Rank: {jack_data['rank']}, Team: {jack_data['team_name']}, Win Rate: {jack_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff_joy = jaxon_data['win_rate']/100 - jack_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff_joy:.4f} (Jaxon {jaxon_data['win_rate']}% - Jack {jack_data['win_rate']}%)")

# Prepare features for prediction
features_joy, home_name_joy, away_name_joy, home_data_joy, away_data_joy = prepare_matchup_features(
    "Jaxon Joy", "Jack Crook", hist_features_joy, inference_pipeline_db, weight_class=149
)

# Make predictions
results_joy = predict_matchup("Jaxon Joy", "Jack Crook", features_joy, home_name_joy, away_name_joy)

# Print detailed information
print_matchup_details(
    "Jaxon Joy", "Jack Crook",
    hist_features_joy, features_joy, results_joy,
    home_name_joy, away_name_joy, home_data_joy, away_data_joy
)


# ============================================
# VERIFICATION OF HOME/AWAY RULES
# ============================================

print("\n" + "="*100)
print("🔍 VERIFYING HOME/AWAY RULES")
print("="*100)

print("\nRule 1: Columbia always home")
print("   Sulayman Bah (Columbia) vs anyone → Sulayman Bah should be home")

# Test Columbia rule
sulayman_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Sulayman Bah'].iloc[0]
print(f"   Sulayman Bah - Team: {sulayman_data['team_name']}")

print("\nRule 2: Otherwise, lower rank is home")
print("   Jaxon Joy (Rank 3) vs Jack Crook (Rank 60) → Jaxon Joy should be home")
print(f"   Jaxon Joy rank: {jaxon_data['rank']}, Jack Crook rank: {jack_data['rank']}")
print(f"   Actual home in test case: {home_name_joy}")

print("\nRule 3: Historical point diff adjusted based on home")
print("   Jaxon Joy vs Jack Crook - Jaxon won by 14")
print(f"   Raw historical (Jaxon's perspective): {hist_features_joy['avg_point_diff_w1']}")
print(f"   Adjusted for home ({home_name_joy}): {features_joy['home_point_diff_rematches']}")



🔍 IVY LEAGUE TEST CASE 1: SIMON RUIZ vs NICK FINE

📜 Raw Historical Features (from Simon's perspective):
   wrestled_before: 1
   home_point_diff_rematches: 3.0
   total_matches: 1
   Simon Ruiz wins: 1
   Nick Fine wins: 0
   Avg point diff (Simon's perspective): 3.0

📊 Wrestler Data from Inference Pipeline:
   Simon Ruiz - Rank: 2, Team: Cornell, Win Rate: 100.0%
   Nick Fine - Rank: 22, Team: Columbia, Win Rate: 76.9%

📊 LOGREG Input (win_rate_diff): 0.2310 (Simon 100.0% - Nick 76.9%)

📌 Home wrestler: Nick Fine (Rank: 22, Team: Columbia)
📌 Away wrestler: Simon Ruiz (Rank: 2, Team: Cornell)

🔍 MATCHUP DETAILS: Simon Ruiz vs Nick Fine

📜 HISTORICAL SUMMARY:
   Previous matches: 1
   Simon Ruiz wins: 1
   Nick Fine wins: 0
   Avg point diff (Simon Ruiz's perspective): 3.0
   Last match: 1/31/2026 (Winner: Simon Ruiz)

🔍 HISTORICAL FEATURES FOR MODELS:
   wrestled_before: 1
   home_point_diff_rematches: -3.0
   home_pinned_away: 0

📊 KEY DIFFERENTIALS (home - away):
   win_rate_diff: 

In [81]:

# ============================================
# COMBINE INTO MASTER LIST
# ============================================

all_wrestler_list = [
    wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149,
    wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184,
    wrestlers_197, wrestlers_285
]

weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

# Dictionary to store all results
all_weight_results = {}

# ============================================
# PREDICTION LOOP FOR IVY LEAGUE
# ============================================

# Loop through each weight class
for weight_idx, (weight, wrestler_list) in enumerate(zip(weights, all_wrestler_list)):
    print("\n" + "="*100)
    print(f"🏋️  PROCESSING IVY LEAGUE {weight}lbs WEIGHT CLASS ({weight_idx+1}/10)")
    print("="*100)

    # Display wrestlers in this weight class
    print(f"\n📋 {weight}lbs Wrestlers:")
    wrestlers_df = inference_pipeline_db[inference_pipeline_db['wrestler_name'].isin(wrestler_list)].copy()

    # Check if all wrestlers were found
    found_wrestlers = wrestlers_df['wrestler_name'].tolist()
    missing = [w for w in wrestler_list if w not in found_wrestlers]

    if missing:
        print(f"⚠️ Warning: {len(missing)} wrestlers not found in database:")
        for w in missing:
            print(f"   - {w}")
        # Remove missing wrestlers from the list
        wrestler_list = [w for w in wrestler_list if w in found_wrestlers]

    if len(wrestlers_df) == 0 or len(wrestler_list) < 2:
        print(f"❌ Not enough wrestlers found for {weight}lbs! Skipping...")
        continue

    print(wrestlers_df[['wrestler_name', 'team_name', 'rank', 'win_rate']].to_string(index=False))

    # Generate all matchup predictions for this weight class
    total_matchups = len(list(combinations(wrestler_list, 2)))
    print(f"\n🔄 Generating predictions for {total_matchups} matchups...")

    weight_predictions = []
    matchup_count = 0

    for w1, w2 in combinations(wrestler_list, 2):
        matchup_count += 1
        if matchup_count % 5 == 0 or matchup_count == total_matchups:
            print(f"   Progress: {matchup_count}/{total_matchups} matchups")

        # Get historical data
        hist_features = get_wrestler_match_history(w1, w2, matches_reference)

        # Prepare features - using the Ivy League version with Columbia priority
        features, home_name, away_name, home_data, away_data = prepare_matchup_features(
            w1, w2, hist_features, inference_pipeline_db, weight_class=weight
        )

        # Make predictions
        results = predict_matchup(w1, w2, features, home_name, away_name)

        # Add weight class
        results['weight_class'] = weight
        weight_predictions.append(results)

    # Create dataframe for this weight class
    weight_df = pd.DataFrame(weight_predictions)

    # Add model agreement column
    weight_df['all_agree'] = (
        (weight_df['logreg_winner'] == weight_df['dt_winner']) &
        (weight_df['dt_winner'] == weight_df['xgb1_winner']) &
        (weight_df['xgb1_winner'] == weight_df['xgb2_winner'])
    )

    # Store results
    all_weight_results[weight] = weight_df

    # ============================================
    # DISPLAY ALL MATCHUPS WITH WINNER PROBABILITIES
    # ============================================
    print(f"\n📊 IVY LEAGUE {weight}lbs - ALL MATCHUP PREDICTIONS")
    print("="*100)

    display_cols = [
        'home_wrestler', 'away_wrestler',
        'logreg_winner', 'logreg_prob',
        'dt_winner', 'dt_prob',
        'xgb1_winner', 'xgb1_prob',
        'xgb2_winner', 'xgb2_prob',
        'all_agree'
    ]

    display_df = weight_df[display_cols].copy()
    display_df.columns = [
        'Home', 'Away',
        'LR_Winner', 'LR_Conf',
        'DT_Winner', 'DT_Conf',
        'X1_Winner', 'X1_Conf',
        'X2_Winner', 'X2_Conf',
        'Agree'
    ]

    # Format confidence as percentage
    for col in ['LR_Conf', 'DT_Conf', 'X1_Conf', 'X2_Conf']:
        display_df[col] = (display_df[col] * 100).round(1).astype(str) + '%'

    print(display_df.to_string(index=False))

    # ============================================
    # QUICK SUMMARY STATS
    # ============================================
    print(f"\n📊 IVY LEAGUE {weight}lbs - QUICK SUMMARY")
    print("-" * 60)

    # Agreement stats
    agreement_count = weight_df['all_agree'].sum()
    print(f"Total Matchups: {len(weight_df)}")
    print(f"All models agree: {agreement_count}/{len(weight_df)} ({agreement_count/len(weight_df)*100:.1f}%)")

    # Calculate average confidence
    weight_df['avg_confidence'] = (
        weight_df['logreg_prob'] +
        weight_df['dt_prob'] +
        weight_df['xgb1_prob'] +
        weight_df['xgb2_prob']
    ) / 4

    # Top 3 most confident predictions
    if len(weight_df) >= 3:
        top_confident = weight_df.nlargest(3, 'avg_confidence')[
            ['home_wrestler', 'away_wrestler', 'avg_confidence']
        ]
        print("\n🔝 Top 3 most confident predictions:")
        for _, row in top_confident.iterrows():
            print(f"   {row['home_wrestler']} vs {row['away_wrestler']}: {row['avg_confidence']*100:.1f}%")

    # Predicted champion by each model
    print("\n🏆 Predicted champions:")
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        if len(weight_df) > 0:
            champ_counts = weight_df[f'{model}_winner'].value_counts()
            if len(champ_counts) > 0:
                champ = champ_counts.index[0]
                champ_wins = champ_counts.values[0]
                champ_pct = (champ_wins / (len(wrestler_list) - 1)) * 100 if len(wrestler_list) > 1 else 0
                print(f"   {model.upper():6}: {champ} ({champ_wins}/{len(wrestler_list)-1} wins, {champ_pct:.1f}%)")

    # Check for unanimous champion
    champions = []
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        if len(weight_df) > 0:
            champ_counts = weight_df[f'{model}_winner'].value_counts()
            if len(champ_counts) > 0:
                champions.append(champ_counts.index[0])

    if len(champions) == 4 and all(c == champions[0] for c in champions):
        print(f"\n✅ UNANIMOUS CHAMPION: {champions[0]}")
    elif len(champions) > 0:
        print(f"\n⚠️ No unanimous champion")
        for model, champ in zip(['logreg', 'dt', 'xgb1', 'xgb2'], champions):
            print(f"   {model.upper():6}: {champ}")

    print("\n" + "="*60)

# ============================================
# OVERALL SUMMARY ACROSS ALL WEIGHT CLASSES
# ============================================
print("\n" + "="*100)
print("📊 IVY LEAGUE OVERALL SUMMARY - ALL WEIGHT CLASSES")
print("="*100)

summary_rows = []
for weight in weights:
    if weight in all_weight_results:
        weight_df = all_weight_results[weight]

        # Agreement stats
        agreement_count = weight_df['all_agree'].sum()
        agreement_pct = agreement_count / len(weight_df) * 100 if len(weight_df) > 0 else 0

        # Champions by model
        logreg_champ = weight_df['logreg_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"
        dt_champ = weight_df['dt_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"
        xgb1_champ = weight_df['xgb1_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"
        xgb2_champ = weight_df['xgb2_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"

        # Check unanimous
        unanimous = (logreg_champ == dt_champ == xgb1_champ == xgb2_champ) and logreg_champ != "N/A"

        summary_rows.append({
            'Weight': f"{weight}lbs",
            'Matchups': len(weight_df),
            'Agreement': f"{agreement_pct:.1f}%",
            'LogReg Champ': logreg_champ[:15] + "..." if len(str(logreg_champ)) > 15 else str(logreg_champ),
            'DT Champ': dt_champ[:15] + "..." if len(str(dt_champ)) > 15 else str(dt_champ),
            'XGB1 Champ': xgb1_champ[:15] + "..." if len(str(xgb1_champ)) > 15 else str(xgb1_champ),
            'XGB2 Champ': xgb2_champ[:15] + "..." if len(str(xgb2_champ)) > 15 else str(xgb2_champ),
            'Unanimous': '✅' if unanimous else '❌'
        })

summary_df = pd.DataFrame(summary_rows)
print("\n", summary_df.to_string(index=False))

# ============================================
# SAVE MASTER SUMMARY
# ============================================
summary_df.to_csv('ivy_league_all_weight_classes_summary.csv', index=False)
print("\n💾 Saved Ivy League master summary to ivy_league_all_weight_classes_summary.csv")

# Combine all predictions into one master file
if len(all_weight_results) > 0:
    all_predictions_master = pd.concat(all_weight_results.values(), ignore_index=True)
    all_predictions_master.to_csv('ivy_league_all_predictions_master.csv', index=False)
    print("💾 Saved all Ivy League predictions to ivy_league_all_predictions_master.csv")

print("\n" + "="*100)
print("✅ IVY LEAGUE ALL WEIGHT CLASS PREDICTIONS COMPLETE")
print("="*100)
print(f"Total weight classes processed: {len(all_weight_results)}")
print(f"Total matchups processed: {sum(len(df) for df in all_weight_results.values())}")


🏋️  PROCESSING IVY LEAGUE 125lbs WEIGHT CLASS (1/10)

📋 125lbs Wrestlers:
       wrestler_name    team_name  rank  win_rate
        Sulayman Bah     Columbia    23      69.2
    Logan Brzozowski      Harvard    45      66.7
      Chase Yasutake        Brown   230       9.1
Marc-Anthony McGowan    Princeton    13      80.0
   Greg Diakomihalis      Cornell    50      33.3
        Davis Motyka Pennsylvania    37      80.0

🔄 Generating predictions for 15 matchups...

📌 Home wrestler: Davis Motyka (Rank: 37, Team: Pennsylvania)
📌 Away wrestler: Greg Diakomihalis (Rank: 50, Team: Cornell)

📌 Home wrestler: Davis Motyka (Rank: 37, Team: Pennsylvania)
📌 Away wrestler: Logan Brzozowski (Rank: 45, Team: Harvard)

📌 Home wrestler: Sulayman Bah (Rank: 23, Team: Columbia)
📌 Away wrestler: Davis Motyka (Rank: 37, Team: Pennsylvania)

📌 Home wrestler: Davis Motyka (Rank: 37, Team: Pennsylvania)
📌 Away wrestler: Chase Yasutake (Rank: 230, Team: Brown)
   Progress: 5/15 matchups

📌 Home wrestler: Ma

# ----- now we will switch to the big ten

In [82]:
# ============================================
# BIG TEN WRESTLER LISTS
# ============================================

print("\n" + "="*100)
print("🏆 BIG TEN CONFERENCE WRESTLERS BY WEIGHT CLASS")
print("="*100)

# 125 lbs
wrestlers_125 = [
    "Luke Lilledahl", "Spencer Moore", "Nic Bouzakis", "Jacob Moran", "Jore Volk",
    "Ayden Smith", "Nicolar Rivera", "Dean Peterson", "Diego Sotelo", "Dedrick Navarro",
    "Kael Lauridsen", "Nick Corday", "Ashton Jackson", "Abram Cline"
]
print("\n125 BIG TEN WRESTLERS")
filter_weights(wrestlers_125)
print()

# 133 lbs
wrestlers_133 = [
    "Marcus Blaze", "Lucas Byrd", "Ben Davino", "Zan Fugitt", "Drake Ayala",
    "Jacob Van Dee", "Sean Spidle", "Braxton Brown", "Dylan Shawver", "Blake Boarman",
    "Caleb Weiand", "Blaine Frazier", "Gauge Botero", "Jager Eisch"
]
print("133 BIG TEN WRESTLERS")
filter_weights(wrestlers_133)
print()

# 141 lbs
wrestlers_141 = [
    "Jesse Mendez", "Brock Hardy", "Vance Vombaur", "Nasir Bailey", "Dylan Ragusin",
    "Greyson Clark", "Braeden Davis", "Billy Dekraker", "Henry Porter", "Joey Olivieri",
    "Dario Lemus", "Danny Pucino", "Carson Exferd", "Jaden Crumpler"
]
print("141 BIG TEN WRESTLERS")
filter_weights(wrestlers_141)
print()

# 149 lbs
wrestlers_149 = [
    "Shayne Van Ness", "Ethan Stiles", "Joseph Zargo", "Lachlan McNeil", "Carter Young",
    "Chance Lamer", "Andrew Clark", "Ryder Block", "Michael Gioffre", "Drew Roberts",
    "Joey Buttler", "Gavin Brown", "Clayton Jones", "August Hibler"
]
print("149 BIG TEN WRESTLERS")
filter_weights(wrestlers_149)
print()

# 157 lbs
wrestlers_157 = [
    "Antrell Taylor", "PJ Duke", "Kannon Webster", "Anthony White", "Cam Catrabone",
    "Charlie Millard", "Brandon Cannon", "Luke Mechler", "Stoney Buell", "Victor Voinovich",
    "Bryce Lowery", "Darius Marines", "Mekhi Neal", "Ty Wilson"
]
print("157 BIG TEN WRESTLERS")
filter_weights(wrestlers_157)
print()

# 165 lbs
wrestlers_165 = [
    "Mitchell Mesenbrink", "Michael Caliendo", "Joey Blaze", "Andrew Sparks", "LJ Araujo",
    "Braeden Scoles", "Andrew Barbosa", "Paddy Gallagher", "Tyler Lillard", "Cody Goebel",
    "Jacob Bostelman", "Justin Gates", "AJ Rodriguez", "Jack Conley"
]
print("165 BIG TEN WRESTLERS")
filter_weights(wrestlers_165)
print()

# 174 lbs
wrestlers_174 = [
    "Christopher Minto", "Levi Haines", "Patrick Kennedy", "Beau Mantanona", "Carson Kharchla",
    "Derek Gilcher", "Ethan Riddle", "Brody Baumann", "Colin Kelly", "Lenny Pinto",
    "Eddie Enright", "Lucas Condon", "Connor O'Neill", "Seth Digby"
]
print("174 BIG TEN WRESTLERS")
filter_weights(wrestlers_174)
print()

# 184 lbs
wrestlers_184 = [
    "Rocco Welsh", "Max McEnelly", "Silas Allred", "Brock Mantanona", "Chris Moore",
    "Dylan Fishback", "Shane Cartagena-Walsh", "Angelo Ferrari", "Sam Goin", "James Rowley",
    "Jesse Perez", "Sepanta Ahanj-Elias", "Ryan Boucher", "Cale Anderson"
]
print("184 BIG TEN WRESTLERS")
filter_weights(wrestlers_184)
print()

# 197 lbs
wrestlers_197 = [
    "Josh Barr", "Camden McDanel", "Remy Cotton", "Branson John", "Luke Geog",
    "Wyatt Ingham", "Kael Wisler", "Gabe Sollars", "Ben Vanadia", "Hayden Walters",
    "Gavin Nelson", "Dylan Connell", "Gabe Arnold", "Alex Smith"
]
print("197 BIG TEN WRESTLERS")
filter_weights(wrestlers_197)
print()

# 285 lbs
wrestlers_285 = [
    "Taye Ghadiali", "AJ Ferrari", "Nick Feldman", "Cole Mirasola", "Braxton Amos",
    "Luke Luffman", "Koy Hopke", "Hunter Catka", "Josh Terrill", "Ben Kueter",
    "Hayden Filipovich", "Joey Schneck", "Gabe Christenson", "Caleb Marzolino"
]
print("285 BIG TEN WRESTLERS")
filter_weights(wrestlers_285)
print()

# ============================================
# COMBINE INTO MASTER LIST FOR PREDICTIONS
# ============================================

all_wrestler_list = [
    wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149,
    wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184,
    wrestlers_197, wrestlers_285
]

weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

print("\n" + "="*100)
print(f"✅ BIG TEN CONFERENCE LOADED: {sum(len(lst) for lst in all_wrestler_list)} total wrestlers")
print("="*100)


🏆 BIG TEN CONFERENCE WRESTLERS BY WEIGHT CLASS

125 BIG TEN WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
29,Ayden Smith,Rutgers,35,8,[125],18,10,8,55.6,1,5.6,0,0,1,9,50.06,4.29,0.24,3.78,RSFR
38,Nicolar Rivera,Wisconsin,15,22,[125],17,13,4,76.5,5,29.4,2,3,0,8,65.88,10.29,5.29,5.68,JR
107,Jore Volk,Minnesota,18,8,[125],16,12,4,75.0,3,18.8,1,2,0,9,45.31,6.31,3.12,5.33,RSJR
161,Spencer Moore,Illinois,16,12,[125],15,9,6,60.0,3,20.0,0,2,1,6,48.80,5.43,2.21,5.41,SR
216,Jacob Moran,Indiana,12,33,[125],14,11,3,78.6,8,57.1,6,2,0,3,62.36,13.46,7.92,8.07,SR
223,Diego Sotelo,Michigan,39,7,[125],14,7,7,50.0,3,21.4,1,2,0,4,44.00,7.17,1.83,8.28,SR
240,Dean Peterson,Iowa,8,5,[125],14,10,4,71.4,2,14.3,1,1,0,8,36.57,6.62,3.08,5.69,SR
241,Luke Lilledahl,Penn State,1,1,[125],14,14,0,100.0,9,64.3,6,2,1,5,46.21,13.92,9.92,5.69,SO
306,Nick Corday,Michigan State,82,60,[125],13,6,7,46.2,1,7.7,0,1,0,5,62.38,4.25,-1.08,7.33,JR
460,Nic Bouzakis,Ohio State,2,2,[125],11,10,1,90.9,7,63.6,3,2,2,3,37.73,11.89,8.56,6.37,JR




133 BIG TEN WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
11,Ben Davino,Ohio State,4,2,[133],19,18,1,94.7,12,63.2,9,2,1,6,63.00,13.17,9.89,5.73,RSFR
43,Zan Fugitt,Wisconsin,12,22,[133],17,13,4,76.5,8,47.1,2,4,2,5,65.76,8.60,4.73,7.44,SO
48,Dylan Shawver,Rutgers,35,8,[133],17,11,6,64.7,5,29.4,2,2,1,6,73.53,7.31,3.12,6.78,SR
56,Lucas Byrd,Illinois,1,12,[133],17,17,0,100.0,5,29.4,4,0,1,12,65.41,9.81,7.06,5.03,SR
79,Blake Boarman,Purdue,43,37,[133],16,8,8,50.0,2,12.5,1,0,1,6,71.44,6.71,-0.93,8.29,SR
85,Drake Ayala,Iowa,10,5,[133],16,9,7,56.2,7,43.8,5,1,1,2,29.25,10.40,3.73,9.90,SR
126,Marcus Blaze,Penn State,3,1,[133],15,15,0,100.0,12,80.0,4,5,3,3,46.73,12.42,9.67,5.16,FR
132,Jacob Van Dee,Nebraska,13,6,[133],15,12,3,80.0,5,33.3,2,2,1,7,63.13,7.50,4.43,6.68,JR
146,Gauge Botero,Michigan,98,7,[133],15,2,13,13.3,1,6.7,1,0,0,1,38.73,4.67,-4.93,8.16,FR
386,Braxton Brown,Maryland,23,59,[133],12,8,4,66.7,2,16.7,1,0,1,6,54.50,5.09,0.91,7.31,SR




141 BIG TEN WRESTLERS
Overall Stats


...



197 BIG TEN WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
10,Camden McDanel,Nebraska,13,6,[197],19,15,4,78.9,5,26.3,2,3,0,10,45.79,8.68,4.37,6.25,SO
39,Wyatt Ingham,Wisconsin,24,22,[197],17,12,5,70.6,8,47.1,3,3,2,4,47.41,9.00,4.47,8.68,RSFR
143,Hayden Walters,Michigan,53,7,[197],15,7,8,46.7,3,20.0,0,2,1,4,51.20,4.64,-0.71,6.70,RSFR
193,Gabe Sollars,Indiana,27,33,[197],14,7,7,50.0,4,28.6,1,1,2,3,39.14,5.75,0.08,7.62,SR
196,Gavin Nelson,Minnesota,37,8,[197],14,4,10,28.6,0,0.0,0,0,0,4,23.21,2.62,-1.54,2.93,FR
211,Remy Cotton,Rutgers,22,8,[197],14,11,3,78.6,6,42.9,1,3,2,5,48.21,8.50,2.00,9.18,SO
228,Luke Geog,Ohio State,25,2,[197],14,8,6,57.1,5,35.7,1,4,0,3,43.86,9.15,2.85,8.40,JR
233,Branson John,Maryland,11,59,[197],14,10,4,71.4,7,50.0,4,1,2,3,60.43,10.08,2.92,11.30,SO
265,Kael Wisler,Michigan State,34,60,[197],13,9,4,69.2,3,23.1,2,1,0,6,68.92,8.15,3.54,7.57,RSSO
293,Ben Vanadia,Purdue,40,37,[197],13,9,4,69.2,6,46.2,1,3,2,3,76.38,9.64,3.09,9.46,SR




285 BIG TEN WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
40,Braxton Amos,Wisconsin,12,22,[285],17,14,3,82.4,10,58.8,1,4,5,4,63.65,7.50,5.58,6.37,RSJR
44,Nick Feldman,Ohio State,5,2,[285],17,13,4,76.5,7,41.2,5,1,1,6,39.47,8.75,5.31,7.43,JR
89,Hunter Catka,Rutgers,23,8,[285],16,10,6,62.5,7,43.8,5,1,1,3,88.44,10.13,5.20,8.30,SR
120,Hayden Filipovich,Purdue,77,37,[285],16,8,8,50.0,0,0.0,0,0,0,8,74.69,3.50,-2.06,6.27,SR
123,Cole Mirasola,Penn State,8,1,[285],15,11,4,73.3,7,46.7,4,1,2,4,53.87,9.08,4.69,8.06,RSFR
134,AJ Ferrari,Nebraska,3,6,[285],15,13,2,86.7,7,46.7,2,4,1,6,35.67,8.29,5.86,5.93,JR
149,Taye Ghadiali,Michigan,4,7,[285],15,14,1,93.3,9,60.0,6,2,1,5,44.27,12.36,8.21,7.38,RSSR
229,Koy Hopke,Minnesota,15,8,[285],14,8,6,57.1,3,21.4,0,3,0,5,35.29,5.43,1.86,5.50,RSFR
517,Gabe Christenson,Northwestern,109,41,[285],10,2,8,20.0,2,20.0,0,2,0,0,51.00,4.78,-5.11,10.09,SR
556,Luke Luffman,Illinois,17,12,[285],10,8,2,80.0,3,30.0,0,3,0,5,60.60,8.10,4.50,6.77,SR





✅ BIG TEN CONFERENCE LOADED: 140 total wrestlers


## Explore test cases

In [83]:
matches_reference.query('home_name == "Brock Hardy" or away_name == "Brock Hardy"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
103,272,141,11/7/2025,217.0,Brock Hardy,3.0,960.0,Braden Basile,32.0,True,WTF,WTF5 18 - 2 6:16,SR,Nebraska,JR,Army West Point
418,220,141,11/15/2025,217.0,Brock Hardy,3.0,639.0,Carter Bailey,44.0,True,WFALL,WFALL 6:11,SR,Nebraska,SR,Lehigh
533,207,141,11/15/2025,217.0,Brock Hardy,3.0,1091.0,Eren Sement,40.0,True,WTF,WTF5 20 - 5 7:00,SR,Nebraska,FR,Michigan
861,189,141,11/16/2025,444.0,Jesse Mendez,2.0,217.0,Brock Hardy,3.0,True,WDEC,WDEC 4 - 1,SR,Ohio State,SR,Nebraska
873,190,141,11/16/2025,592.0,Sergio Vega,1.0,217.0,Brock Hardy,3.0,True,WMD,WMD 13 - 2,FR,Oklahoma State,SR,Nebraska
1289,148,141,12/5/2025,217.0,Brock Hardy,3.0,1056.0,Khimari Manns,90.0,True,WFALL,WFALL 2:32,SR,Nebraska,FR,Brown
1300,149,141,12/5/2025,929.0,Zeke Seltzer,62.0,217.0,Brock Hardy,3.0,False,LTF,LTF5 19 - 4 5:40,JR,Missouri,SR,Nebraska
1822,99,141,12/19/2025,217.0,Brock Hardy,3.0,645.0,Luke Simcox,26.0,True,WDEC,WDEC 5 - 0,SR,Nebraska,RSFR,North Carolina
2126,60,141,12/21/2025,217.0,Brock Hardy,3.0,592.0,Sergio Vega,1.0,False,LFALL,LFALL 1:47,SR,Nebraska,FR,Oklahoma State
2351,48,141,1/3/2026,217.0,Brock Hardy,3.0,32.0,Cory Land,16.0,True,WDEC,WDEC 5 - 1,SR,Nebraska,JR,Northern Iowa


In [84]:
display(inference_pipeline_db.query('wrestler_name == "Brock Hardy"'))
display(inference_pipeline_db.query('wrestler_name == "Braeden Davis"'))
#

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
21,Brock Hardy,Nebraska,3,6,[141],18,14,4,77.8,10,55.6,6,0,4,4,35.39,10.54,6.0,9.93,SR


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
674,Braeden Davis,Penn State,11,1,[141],8,5,3,62.5,4,50.0,0,4,0,1,33.5,11.57,4.86,10.29,JR


In [85]:
# ============================================
# BIG TEN TEST CASES
# ============================================

print("\n" + "="*100)
print("🏆 BIG TEN CONFERENCE TEST CASES")
print("="*100)

# ============================================
# CASE 1: BEN DAVINO vs DRAKE AYALA
# ============================================

print("\n" + "="*80)
print("🔍 TEST CASE 1: BEN DAVINO vs DRAKE AYALA")
print("="*80)

# Get their match history
hist_features_ben_drake = get_wrestler_match_history(
    "Ben Davino", "Drake Ayala", matches_reference
)

print(f"\n📜 Raw Historical Features (from Ben Davino's perspective):")
print(f"   wrestled_before: {hist_features_ben_drake['wrestled_before']}")
print(f"   total_matches: {hist_features_ben_drake['total_matches']}")
print(f"   Ben Davino wins: {hist_features_ben_drake['w1_wins']}")
print(f"   Drake Ayala wins: {hist_features_ben_drake['w2_wins']}")
print(f"   Avg point diff (Ben's perspective): {hist_features_ben_drake['avg_point_diff_w1']}")
print(f"   Last match: {hist_features_ben_drake['last_match_date']} (Winner: {hist_features_ben_drake['last_winner']})")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
ben_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Ben Davino'].iloc[0]
drake_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Drake Ayala'].iloc[0]

print(f"   Ben Davino - Rank: {ben_data['rank']}, Team: {ben_data['team_name']}, Win Rate: {ben_data['win_rate']}%")
print(f"   Drake Ayala - Rank: {drake_data['rank']}, Team: {drake_data['team_name']}, Win Rate: {drake_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff = ben_data['win_rate']/100 - drake_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff:.4f} (Ben {ben_data['win_rate']}% - Drake {drake_data['win_rate']}%)")

# Prepare features for prediction - Penn State forced home (but neither is Penn State, so rank decides)
features_ben_drake, home_name_ben_drake, away_name_ben_drake, home_data_ben_drake, away_data_ben_drake = prepare_matchup_features(
    "Ben Davino", "Drake Ayala", hist_features_ben_drake, inference_pipeline_db, team_name="Penn State", weight_class=133
)

# Make predictions
results_ben_drake = predict_matchup("Ben Davino", "Drake Ayala", features_ben_drake, home_name_ben_drake, away_name_ben_drake)

# Print detailed information
print_matchup_details(
    "Ben Davino", "Drake Ayala",
    hist_features_ben_drake, features_ben_drake, results_ben_drake,
    home_name_ben_drake, away_name_ben_drake, home_data_ben_drake, away_data_ben_drake
)

# Verification of expected values
print("\n" + "="*60)
print("✅ EXPECTED VALUES VERIFICATION")
print("="*60)
print(f"Expected wrestled_before: 1 → Actual: {features_ben_drake['wrestled_before']}")
print(f"Expected home_point_diff_rematches (Ben's avg): {(6+2)/2} → Actual: {features_ben_drake['home_point_diff_rematches']}")
print(f"Expected home_pinned_away: 0 → Actual: {features_ben_drake['home_pinned_away']}")


# ============================================
# CASE 2: BEN DAVINO vs MARCUS BLAZE
# ============================================

print("\n" + "="*80)
print("🔍 TEST CASE 2: BEN DAVINO vs MARCUS BLAZE")
print("="*80)

# Get their match history
hist_features_ben_marcus = get_wrestler_match_history(
    "Ben Davino", "Marcus Blaze", matches_reference
)

print(f"\n📜 Raw Historical Features (from Ben Davino's perspective):")
print(f"   wrestled_before: {hist_features_ben_marcus['wrestled_before']}")
print(f"   total_matches: {hist_features_ben_marcus['total_matches']}")
print(f"   Ben Davino wins: {hist_features_ben_marcus['w1_wins']}")
print(f"   Marcus Blaze wins: {hist_features_ben_marcus['w2_wins']}")
print(f"   Avg point diff (Ben's perspective): {hist_features_ben_marcus['avg_point_diff_w1']}")
print(f"   Last match: {hist_features_ben_marcus['last_match_date']} (Winner: {hist_features_ben_marcus['last_winner']})")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
ben_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Ben Davino'].iloc[0]
marcus_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Marcus Blaze'].iloc[0]

print(f"   Ben Davino - Rank: {ben_data['rank']}, Team: {ben_data['team_name']}, Win Rate: {ben_data['win_rate']}%")
print(f"   Marcus Blaze - Rank: {marcus_data['rank']}, Team: {marcus_data['team_name']}, Win Rate: {marcus_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff = ben_data['win_rate']/100 - marcus_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff:.4f} (Ben {ben_data['win_rate']}% - Marcus {marcus_data['win_rate']}%)")

# Prepare features for prediction - Penn State forced home (Marcus is Penn State)
features_ben_marcus, home_name_ben_marcus, away_name_ben_marcus, home_data_ben_marcus, away_data_ben_marcus = prepare_matchup_features(
    "Ben Davino", "Marcus Blaze", hist_features_ben_marcus, inference_pipeline_db, team_name="Penn State", weight_class=133
)

# Make predictions
results_ben_marcus = predict_matchup("Ben Davino", "Marcus Blaze", features_ben_marcus, home_name_ben_marcus, away_name_ben_marcus)

# Print detailed information
print_matchup_details(
    "Ben Davino", "Marcus Blaze",
    hist_features_ben_marcus, features_ben_marcus, results_ben_marcus,
    home_name_ben_marcus, away_name_ben_marcus, home_data_ben_marcus, away_data_ben_marcus
)

# Verification of expected values
print("\n" + "="*60)
print("✅ EXPECTED VALUES VERIFICATION")
print("="*60)
print(f"Expected wrestled_before: 1 → Actual: {features_ben_marcus['wrestled_before']}")
print(f"Expected home_point_diff_rematches (Marcus's avg from his perspective): 1 → Actual: {features_ben_marcus['home_point_diff_rematches']}")
print(f"Expected home_pinned_away: 0 → Actual: {features_ben_marcus['home_pinned_away']}")
print(f"Expected home wrestler (Marcus - Penn State): Marcus Blaze → Actual: {home_name_ben_marcus}")


# ============================================
# CASE 3: AJ FERRARI vs NICK FELDMAN
# ============================================

print("\n" + "="*80)
print("🔍 TEST CASE 3: AJ FERRARI vs NICK FELDMAN")
print("="*80)

# Get their match history
hist_features_aj_nick = get_wrestler_match_history(
    "AJ Ferrari", "Nick Feldman", matches_reference
)

print(f"\n📜 Raw Historical Features (from AJ Ferrari's perspective):")
print(f"   wrestled_before: {hist_features_aj_nick['wrestled_before']}")
print(f"   total_matches: {hist_features_aj_nick['total_matches']}")
print(f"   AJ Ferrari wins: {hist_features_aj_nick['w1_wins']}")
print(f"   Nick Feldman wins: {hist_features_aj_nick['w2_wins']}")
print(f"   Avg point diff (AJ's perspective): {hist_features_aj_nick['avg_point_diff_w1']}")
print(f"   Last match: {hist_features_aj_nick['last_match_date']} (Winner: {hist_features_aj_nick['last_winner']})")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
aj_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'AJ Ferrari'].iloc[0]
nick_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Nick Feldman'].iloc[0]

print(f"   AJ Ferrari - Rank: {aj_data['rank']}, Team: {aj_data['team_name']}, Win Rate: {aj_data['win_rate']}%")
print(f"   Nick Feldman - Rank: {nick_data['rank']}, Team: {nick_data['team_name']}, Win Rate: {nick_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff = aj_data['win_rate']/100 - nick_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff:.4f} (AJ {aj_data['win_rate']}% - Nick {nick_data['win_rate']}%)")

# Prepare features for prediction - Penn State forced home (neither is Penn State, so rank decides)
features_aj_nick, home_name_aj_nick, away_name_aj_nick, home_data_aj_nick, away_data_aj_nick = prepare_matchup_features(
    "AJ Ferrari", "Nick Feldman", hist_features_aj_nick, inference_pipeline_db, team_name="Penn State", weight_class=285
)

# Make predictions
results_aj_nick = predict_matchup("AJ Ferrari", "Nick Feldman", features_aj_nick, home_name_aj_nick, away_name_aj_nick)

# Print detailed information
print_matchup_details(
    "AJ Ferrari", "Nick Feldman",
    hist_features_aj_nick, features_aj_nick, results_aj_nick,
    home_name_aj_nick, away_name_aj_nick, home_data_aj_nick, away_data_aj_nick
)

# Verification of expected values
print("\n" + "="*60)
print("✅ EXPECTED VALUES VERIFICATION")
print("="*60)
print(f"Expected wrestled_before: 2 → Actual: {features_aj_nick['wrestled_before']}")
print(f"Expected avg point diff from home's perspective: {(1+3)/2 if home_name_aj_nick == 'Nick Feldman' else -((1+3)/2)} → Actual: {features_aj_nick['home_point_diff_rematches']}")
print(f"Expected home_pinned_away: 0 → Actual: {features_aj_nick['home_pinned_away']}")


# ============================================
# CASE 4: BROCK HARDY vs BRAEDEN DAVIS
# ============================================

print("\n" + "="*80)
print("🔍 TEST CASE 4: BROCK HARDY vs BRAEDEN DAVIS")
print("="*80)

# Get their match history
hist_features_brock_braeden = get_wrestler_match_history(
    "Brock Hardy", "Braeden Davis", matches_reference
)

print(f"\n📜 Raw Historical Features (from Brock Hardy's perspective):")
print(f"   wrestled_before: {hist_features_brock_braeden['wrestled_before']}")
print(f"   total_matches: {hist_features_brock_braeden['total_matches']}")
print(f"   Brock Hardy wins: {hist_features_brock_braeden['w1_wins']}")
print(f"   Braeden Davis wins: {hist_features_brock_braeden['w2_wins']}")
print(f"   Avg point diff (Brock's perspective): {hist_features_brock_braeden['avg_point_diff_w1']}")
print(f"   Pins by Brock: {hist_features_brock_braeden['pins_by_w1']}")
print(f"   Pins by Braeden: {hist_features_brock_braeden['pins_by_w2']}")
print(f"   Last match: {hist_features_brock_braeden['last_match_date']} (Winner: {hist_features_brock_braeden['last_winner']})")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
brock_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Brock Hardy'].iloc[0]
braeden_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Braeden Davis'].iloc[0]

print(f"   Brock Hardy - Rank: {brock_data['rank']}, Team: {brock_data['team_name']}, Win Rate: {brock_data['win_rate']}%")
print(f"   Braeden Davis - Rank: {braeden_data['rank']}, Team: {braeden_data['team_name']}, Win Rate: {braeden_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff = brock_data['win_rate']/100 - braeden_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff:.4f} (Brock {brock_data['win_rate']}% - Braeden {braeden_data['win_rate']}%)")

# Prepare features for prediction - Penn State forced home (Braeden is Penn State)
features_brock_braeden, home_name_brock_braeden, away_name_brock_braeden, home_data_brock_braeden, away_data_brock_braeden = prepare_matchup_features(
    "Brock Hardy", "Braeden Davis", hist_features_brock_braeden, inference_pipeline_db, team_name="Penn State", weight_class=141
)

# Make predictions
results_brock_braeden = predict_matchup("Brock Hardy", "Braeden Davis", features_brock_braeden, home_name_brock_braeden, away_name_brock_braeden)

# Print detailed information
print_matchup_details(
    "Brock Hardy", "Braeden Davis",
    hist_features_brock_braeden, features_brock_braeden, results_brock_braeden,
    home_name_brock_braeden, away_name_brock_braeden, home_data_brock_braeden, away_data_brock_braeden
)

# Verification of expected values
print("\n" + "="*60)
print("✅ EXPECTED VALUES VERIFICATION")
print("="*60)
print(f"Expected wrestled_before: 1 → Actual: {features_brock_braeden['wrestled_before']}")
print(f"Expected home_point_diff_rematches (pin means no points, so 0 from winner's perspective): 0 → Actual: {features_brock_braeden['home_point_diff_rematches']}")
print(f"Expected home_pinned_away: 1 (since Brock pinned Braeden and Braeden is home) → Actual: {features_brock_braeden['home_pinned_away']}")
print(f"Expected home wrestler (Braeden - Penn State): Braeden Davis → Actual: {home_name_brock_braeden}")

print("\n" + "="*100)
print("✅ BIG TEN TEST CASES COMPLETE")
print("="*100)


🏆 BIG TEN CONFERENCE TEST CASES

🔍 TEST CASE 1: BEN DAVINO vs DRAKE AYALA

📜 Raw Historical Features (from Ben Davino's perspective):
   wrestled_before: 1
   total_matches: 2
   Ben Davino wins: 2
   Drake Ayala wins: 0
   Avg point diff (Ben's perspective): 4.0
   Last match: 2/6/2026 (Winner: Ben Davino)

📊 Wrestler Data from Inference Pipeline:
   Ben Davino - Rank: 4, Team: Ohio State, Win Rate: 94.7%
   Drake Ayala - Rank: 10, Team: Iowa, Win Rate: 56.2%

📊 LOGREG Input (win_rate_diff): 0.3850 (Ben 94.7% - Drake 56.2%)

📌 Home wrestler: Ben Davino (Rank: 4, Team: Ohio State)
📌 Away wrestler: Drake Ayala (Rank: 10, Team: Iowa)

🔍 MATCHUP DETAILS: Ben Davino vs Drake Ayala

📜 HISTORICAL SUMMARY:
   Previous matches: 2
   Ben Davino wins: 2
   Drake Ayala wins: 0
   Avg point diff (Ben Davino's perspective): 4.0
   Last match: 2/6/2026 (Winner: Ben Davino)

🔍 HISTORICAL FEATURES FOR MODELS:
   wrestled_before: 1
   home_point_diff_rematches: 4.0
   home_pinned_away: 0

📊 KEY DIFFER

# Make Predictions

In [86]:
# ============================================
# BIG TEN WRESTLER LISTS
# ============================================

print("\n" + "="*100)
print("🏆 BIG TEN CONFERENCE - FULL PREDICTION LOOP")
print("="*100)

# # 125 lbs
# wrestlers_125 = [
#     "Luke Lilledahl", "Spencer Moore", "Nic Bouzakis", "Jacob Moran", "Jore Volk",
#     "Ayden Smith", "Nicolar Rivera", "Dean Peterson", "Diego Sotelo", "Dedrick Navarro",
#     "Kael Lauridsen", "Nick Corday", "Ashton Jackson", "Abram Cline"
# ]

# # 133 lbs
# wrestlers_133 = [
#     "Marcus Blaze", "Lucas Byrd", "Ben Davino", "Zan Fugitt", "Drake Ayala",
#     "Jacob Van Dee", "Sean Spidle", "Braxton Brown", "Dylan Shawver", "Blake Boarman",
#     "Caleb Weiand", "Blaine Frazier", "Gauge Botero", "Jager Eisch"
# ]

# # 141 lbs
# wrestlers_141 = [
#     "Jesse Mendez", "Brock Hardy", "Vance VomBaur", "Nasir Bailey", "Dylan Ragusin",
#     "Greyson Clark", "Braeden Davis", "Billy Dekraker", "Henry Porter", "Joseph Olivieri",
#     "Dario Lemus", "Danny Pucino", "Carson Exferd", "Jaden Crumpler"
# ]

# # 149 lbs
# wrestlers_149 = [
#     "Shayne Van Ness", "Ethan Stiles", "Joseph Zargo", "Lachlan McNeil", "Carter Young",
#     "Chance Lamer", "Andrew Clark", "Ryder Block", "Michael Gioffre", "Drew Roberts",
#     "Joey Buttler", "Gavin Brown", "Clayton Jones", "August Hibler"
# ]

# # 157 lbs
# wrestlers_157 = [
#     "Antrell Taylor", "PJ Duke", "Kannon Webster", "Anthony White", "Cameron Catrabone",
#     "Charlie Millard", "Brandon Cannon", "Luke Mechler", "Stoney Buell", "Victor Voinovich III",
#     "Bryce Lowery", "Darius Marines", "Mekhi Neal", "Ty Wilson"
# ]

# # 165 lbs
# wrestlers_165 = [
#     "Mitchell Mesenbrink", "Michael Caliendo", "Joey Blaze", "Andrew Sparks", "LJ Araujo",
#     "Braeden Scoles", "Andrew Barbosa", "Paddy Gallagher", "Tyler Lillard", "Cody Goebel",
#     "Jacob Bostelman", "Justin Gates", "AJ Rodrigues", "Jack Conley"
# ]

# # 174 lbs
# wrestlers_174 = [
#     "Christopher Minto", "Levi Haines", "Patrick Kennedy", "Beau Mantanona", "Carson Kharchla",
#     "Derek Gilcher", "Ethan Riddle", "Brody Baumann", "Colin Kelly", "Lenny Pinto",
#     "Eddie Enright", "Luke Condon", "Connor O'Neil", "Seth Digby"
# ]

# # 184 lbs
# wrestlers_184 = [
#     "Rocco Welsh", "Max McEnelly", "Silas Allred", "Brock Mantanona", "Chris Moore",
#     "Dylan Fishback", "Shane Cartagena-Walsh", "Angelo Ferrari", "Sam Goin", "James Rowley",
#     "Jesse Perez", "Sepanta Ahanj-Elias", "Ryan Boucher", "Cale Anderson"
# ]

# # 197 lbs
# wrestlers_197 = [
#     "Josh Barr", "Camden McDanel", "Remy Cotton", "Branson John", "Luke Geog",
#     "Wyatt Ingham", "Kael Wisler", "Gabe Sollars", "Ben Vanadia", "Hayden Walters",
#     "Gavin Nelson", "Dylan Connell", "Gabe Arnold", "Alex Smith"
# ]

# # 285 lbs
# wrestlers_285 = [
#     "Taye Ghadiali", "AJ Ferrari", "Nick Feldman", "Cole Mirasola", "Braxton Amos",
#     "Luke Luffman", "Koy Hopke", "Hunter Catka", "Josh Terrill", "Ben Kueter",
#     "Hayden Filipovich", "Joey Schneck", "Gabe Christenson", "Caleb Marzolino"
# ]

# ============================================
# COMBINE INTO MASTER LIST
# ============================================

all_wrestler_list = [
    wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149,
    wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184,
    wrestlers_197, wrestlers_285
]

weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

# Dictionary to store all results
all_weight_results = {}

# ============================================
# PREDICTION LOOP FOR BIG TEN
# ============================================

# Loop through each weight class
for weight_idx, (weight, wrestler_list) in enumerate(zip(weights, all_wrestler_list)):
    print("\n" + "="*100)
    print(f"🏋️  PROCESSING BIG TEN {weight}lbs WEIGHT CLASS ({weight_idx+1}/10)")
    print("="*100)

    # Display wrestlers in this weight class
    print(f"\n📋 {weight}lbs Wrestlers:")
    wrestlers_df = inference_pipeline_db[inference_pipeline_db['wrestler_name'].isin(wrestler_list)].copy()

    # Check if all wrestlers were found
    found_wrestlers = wrestlers_df['wrestler_name'].tolist()
    missing = [w for w in wrestler_list if w not in found_wrestlers]

    if missing:
        print(f"⚠️ Warning: {len(missing)} wrestlers not found in database:")
        for w in missing:
            print(f"   - {w}")
        # Remove missing wrestlers from the list
        wrestler_list = [w for w in wrestler_list if w in found_wrestlers]

    if len(wrestlers_df) == 0 or len(wrestler_list) < 2:
        print(f"❌ Not enough wrestlers found for {weight}lbs! Skipping...")
        continue

    print(wrestlers_df[['wrestler_name', 'team_name', 'rank', 'win_rate']].to_string(index=False))

    # Generate all matchup predictions for this weight class
    total_matchups = len(list(combinations(wrestler_list, 2)))
    print(f"\n🔄 Generating predictions for {total_matchups} matchups...")

    weight_predictions = []
    matchup_count = 0

    for w1, w2 in combinations(wrestler_list, 2):
        matchup_count += 1
        if matchup_count % 10 == 0 or matchup_count == total_matchups:
            print(f"   Progress: {matchup_count}/{total_matchups} matchups")

        # Get historical data
        hist_features = get_wrestler_match_history(w1, w2, matches_reference)

        # Prepare features - Penn State forced home for Big Ten
        features, home_name, away_name, home_data, away_data = prepare_matchup_features(
            w1, w2, hist_features, inference_pipeline_db, team_name="Penn State", weight_class=weight
        )

        # Make predictions
        results = predict_matchup(w1, w2, features, home_name, away_name)

        # Add weight class
        results['weight_class'] = weight
        weight_predictions.append(results)

    # Create dataframe for this weight class
    weight_df = pd.DataFrame(weight_predictions)

    # Add model agreement column
    weight_df['all_agree'] = (
        (weight_df['logreg_winner'] == weight_df['dt_winner']) &
        (weight_df['dt_winner'] == weight_df['xgb1_winner']) &
        (weight_df['xgb1_winner'] == weight_df['xgb2_winner'])
    )

    # Store results
    all_weight_results[weight] = weight_df

    # ============================================
    # DISPLAY ALL MATCHUPS WITH WINNER PROBABILITIES
    # ============================================
    print(f"\n📊 BIG TEN {weight}lbs - ALL MATCHUP PREDICTIONS")
    print("="*100)

    display_cols = [
        'home_wrestler', 'away_wrestler',
        'logreg_winner', 'logreg_prob',
        'dt_winner', 'dt_prob',
        'xgb1_winner', 'xgb1_prob',
        'xgb2_winner', 'xgb2_prob',
        'all_agree'
    ]

    display_df = weight_df[display_cols].copy()
    display_df.columns = [
        'Home', 'Away',
        'LR_Winner', 'LR_Conf',
        'DT_Winner', 'DT_Conf',
        'X1_Winner', 'X1_Conf',
        'X2_Winner', 'X2_Conf',
        'Agree'
    ]

    # Format confidence as percentage
    for col in ['LR_Conf', 'DT_Conf', 'X1_Conf', 'X2_Conf']:
        display_df[col] = (display_df[col] * 100).round(1).astype(str) + '%'

    # Show first 20 rows or all if less
    if len(display_df) > 20:
        print(display_df.head(20).to_string(index=False))
        print(f"... and {len(display_df) - 20} more matchups")
    else:
        print(display_df.to_string(index=False))

    # ============================================
    # QUICK SUMMARY STATS
    # ============================================
    print(f"\n📊 BIG TEN {weight}lbs - QUICK SUMMARY")
    print("-" * 60)

    # Agreement stats
    agreement_count = weight_df['all_agree'].sum()
    print(f"Total Matchups: {len(weight_df)}")
    print(f"All models agree: {agreement_count}/{len(weight_df)} ({agreement_count/len(weight_df)*100:.1f}%)")

    # Calculate average confidence
    weight_df['avg_confidence'] = (
        weight_df['logreg_prob'] +
        weight_df['dt_prob'] +
        weight_df['xgb1_prob'] +
        weight_df['xgb2_prob']
    ) / 4

    # Top 3 most confident predictions
    if len(weight_df) >= 3:
        top_confident = weight_df.nlargest(3, 'avg_confidence')[
            ['home_wrestler', 'away_wrestler', 'avg_confidence']
        ]
        print("\n🔝 Top 3 most confident predictions:")
        for _, row in top_confident.iterrows():
            print(f"   {row['home_wrestler']} vs {row['away_wrestler']}: {row['avg_confidence']*100:.1f}%")

    # Predicted champion by each model
    print("\n🏆 Predicted champions:")
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        if len(weight_df) > 0:
            champ_counts = weight_df[f'{model}_winner'].value_counts()
            if len(champ_counts) > 0:
                champ = champ_counts.index[0]
                champ_wins = champ_counts.values[0]
                champ_pct = (champ_wins / (len(wrestler_list) - 1)) * 100 if len(wrestler_list) > 1 else 0
                print(f"   {model.upper():6}: {champ} ({champ_wins}/{len(wrestler_list)-1} wins, {champ_pct:.1f}%)")

    # Check for unanimous champion
    champions = []
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        if len(weight_df) > 0:
            champ_counts = weight_df[f'{model}_winner'].value_counts()
            if len(champ_counts) > 0:
                champions.append(champ_counts.index[0])

    if len(champions) == 4 and all(c == champions[0] for c in champions):
        print(f"\n✅ UNANIMOUS CHAMPION: {champions[0]}")
    elif len(champions) > 0:
        print(f"\n⚠️ No unanimous champion")
        for model, champ in zip(['logreg', 'dt', 'xgb1', 'xgb2'], champions):
            print(f"   {model.upper():6}: {champ}")

    print("\n" + "="*60)

# ============================================
# OVERALL SUMMARY ACROSS ALL WEIGHT CLASSES
# ============================================
print("\n" + "="*100)
print("📊 BIG TEN OVERALL SUMMARY - ALL WEIGHT CLASSES")
print("="*100)

summary_rows = []
for weight in weights:
    if weight in all_weight_results:
        weight_df = all_weight_results[weight]

        # Agreement stats
        agreement_count = weight_df['all_agree'].sum()
        agreement_pct = agreement_count / len(weight_df) * 100 if len(weight_df) > 0 else 0

        # Champions by model
        logreg_champ = weight_df['logreg_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"
        dt_champ = weight_df['dt_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"
        xgb1_champ = weight_df['xgb1_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"
        xgb2_champ = weight_df['xgb2_winner'].value_counts().index[0] if len(weight_df) > 0 else "N/A"

        # Check unanimous
        unanimous = (logreg_champ == dt_champ == xgb1_champ == xgb2_champ) and logreg_champ != "N/A"

        summary_rows.append({
            'Weight': f"{weight}lbs",
            'Matchups': len(weight_df),
            'Agreement': f"{agreement_pct:.1f}%",
            'LogReg Champ': logreg_champ[:15] + "..." if len(str(logreg_champ)) > 15 else str(logreg_champ),
            'DT Champ': dt_champ[:15] + "..." if len(str(dt_champ)) > 15 else str(dt_champ),
            'XGB1 Champ': xgb1_champ[:15] + "..." if len(str(xgb1_champ)) > 15 else str(xgb1_champ),
            'XGB2 Champ': xgb2_champ[:15] + "..." if len(str(xgb2_champ)) > 15 else str(xgb2_champ),
            'Unanimous': '✅' if unanimous else '❌'
        })

summary_df = pd.DataFrame(summary_rows)
print("\n", summary_df.to_string(index=False))

# ============================================
# SAVE MASTER SUMMARY
# ============================================
summary_df.to_csv('big_ten_all_weight_classes_summary.csv', index=False)
print("\n💾 Saved Big Ten master summary to big_ten_all_weight_classes_summary.csv")

# Combine all predictions into one master file
if len(all_weight_results) > 0:
    all_predictions_master = pd.concat(all_weight_results.values(), ignore_index=True)
    all_predictions_master.to_csv('big_ten_all_predictions_master.csv', index=False)
    print("💾 Saved all Big Ten predictions to big_ten_all_predictions_master.csv")

print("\n" + "="*100)
print("✅ BIG TEN ALL WEIGHT CLASS PREDICTIONS COMPLETE")
print("="*100)
print(f"Total weight classes processed: {len(all_weight_results)}")
print(f"Total matchups processed: {sum(len(df) for df in all_weight_results.values())}")


🏆 BIG TEN CONFERENCE - FULL PREDICTION LOOP

🏋️  PROCESSING BIG TEN 125lbs WEIGHT CLASS (1/10)

📋 125lbs Wrestlers:
  wrestler_name      team_name  rank  win_rate
    Ayden Smith        Rutgers    35      55.6
 Nicolar Rivera      Wisconsin    15      76.5
      Jore Volk      Minnesota    18      75.0
  Spencer Moore       Illinois    16      60.0
    Jacob Moran        Indiana    12      78.6
   Diego Sotelo       Michigan    39      50.0
  Dean Peterson           Iowa     8      71.4
 Luke Lilledahl     Penn State     1     100.0
    Nick Corday Michigan State    82      46.2
   Nic Bouzakis     Ohio State     2      90.9
    Abram Cline       Maryland   125      10.0
 Kael Lauridsen       Nebraska    47      30.0
Dedrick Navarro   Northwestern    38      60.0
 Ashton Jackson         Purdue    96      22.2

🔄 Generating predictions for 91 matchups...

📌 Home wrestler: Luke Lilledahl (Rank: 1, Team: Penn State)
📌 Away wrestler: Spencer Moore (Rank: 16, Team: Illinois)

📌 Home wrestl

In [87]:
# ============================================
# FUNCTION TO FIND MISSING WRESTLERS
# ============================================

def find_missing_wrestlers(wrestler_lists, weight_classes, inference_db):
    """
    Identify which wrestlers from each weight class are missing from the inference database.

    Parameters:
    - wrestler_lists: list of lists containing wrestler names by weight class
    - weight_classes: list of weight class numbers
    - inference_db: the inference pipeline dataframe

    Returns:
    - missing_dict: dictionary with weight class as key and list of missing wrestlers as value
    - found_dict: dictionary with weight class as key and list of found wrestlers as value
    """

    print("\n" + "="*100)
    print("🔍 IDENTIFYING MISSING BIG TEN WRESTLERS")
    print("="*100)

    missing_dict = {}
    found_dict = {}
    all_missing = []

    for weight_idx, (weight, wrestler_list) in enumerate(zip(weight_classes, wrestler_lists)):
        print(f"\n📊 {weight}lbs - Checking {len(wrestler_list)} wrestlers...")

        # Find which wrestlers are in the database
        found = inference_db[inference_db['wrestler_name'].isin(wrestler_list)]['wrestler_name'].tolist()
        missing = [w for w in wrestler_list if w not in found]

        found_dict[weight] = found
        missing_dict[weight] = missing
        all_missing.extend(missing)

        # Print results for this weight class
        print(f"   ✅ Found: {len(found)}/{len(wrestler_list)} wrestlers")
        if missing:
            print(f"   ❌ Missing ({len(missing)}):")
            for w in missing:
                # Try to find similar names for suggestions
                similar = inference_db[inference_db['wrestler_name'].str.contains(w.split()[-1] if ' ' in w else w, na=False)]['wrestler_name'].tolist()
                print(f"      - {w}")
                if similar:
                    print(f"        → Similar names in DB: {similar[:3]}")

    print("\n" + "="*100)
    print(f"📋 SUMMARY: {len(all_missing)} total missing wrestlers across all weight classes")
    print("="*100)

    return missing_dict, found_dict


# ============================================
# FUNCTION TO SUGGEST CORRECTIONS
# ============================================

def suggest_name_corrections(missing_dict, inference_db):
    """
    Suggest possible corrections for missing wrestler names.
    """
    print("\n💡 NAME CORRECTION SUGGESTIONS:")
    print("-" * 60)

    all_suggestions = {}

    for weight, missing_list in missing_dict.items():
        if not missing_list:
            continue

        print(f"\n{weight}lbs:")
        weight_suggestions = {}

        for missing_name in missing_list:
            # Split name to get last name
            parts = missing_name.split()
            if len(parts) > 1:
                last_name = parts[-1]
                # Find similar last names in database
                similar = inference_db[inference_db['wrestler_name'].str.contains(last_name, case=False, na=False)]
                if len(similar) > 0:
                    suggestions = similar['wrestler_name'].tolist()
                    weight_suggestions[missing_name] = suggestions
                    print(f"   '{missing_name}' → maybe: {suggestions}")
                else:
                    print(f"   '{missing_name}' → no close matches found")
            else:
                print(f"   '{missing_name}' → single name, can't parse last name")

        all_suggestions[weight] = weight_suggestions

    return all_suggestions


# ============================================
# RUN THE ANALYSIS
# ============================================

# Your Big Ten wrestler lists
big_ten_lists = [
    wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149,
    wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184,
    wrestlers_197, wrestlers_285
]
weight_classes = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

# Find missing wrestlers
missing_dict, found_dict = find_missing_wrestlers(big_ten_lists, weight_classes, inference_pipeline_db)

# Get correction suggestions
suggestions = suggest_name_corrections(missing_dict, inference_pipeline_db)

# ============================================
# CREATE UPDATED LISTS (optional)
# ============================================

print("\n" + "="*100)
print("📝 UPDATED WRESTLER LISTS (with only found wrestlers)")
print("="*100)

updated_lists = []
for weight in weight_classes:
    updated_list = found_dict[weight]
    updated_lists.append(updated_list)
    print(f"\n{weight}lbs ({len(updated_list)} wrestlers):")
    for w in updated_list:
        print(f"   - {w}")

# ============================================
# CHECK SPECIFIC NAMES THAT MIGHT BE PROBLEMATIC
# ============================================

print("\n" + "="*100)
print("🔎 CHECKING SPECIFIC NAME PATTERNS")
print("="*100)

# Common naming issues to check
problematic_patterns = [
    "Luke Lilledahl",  # Might be "Lilledahl, Luke" format
    "PJ Duke",         # Might have period after PJ
    "AJ Ferrari",      # Might have period after AJ
    "Joey Buttler",    # Might be misspelled
    "Jager Eisch",     # Uncommon name
]

for name in problematic_patterns:
    print(f"\nChecking '{name}':")

    # Exact match
    exact = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == name]
    if len(exact) > 0:
        print(f"   ✅ Found exact match!")
        continue

    # Try contains
    parts = name.split()
    if len(parts) > 1:
        last_name = parts[-1]
        contains = inference_pipeline_db[inference_pipeline_db['wrestler_name'].str.contains(last_name, case=False, na=False)]
        if len(contains) > 0:
            print(f"   🔍 Found by last name '{last_name}':")
            for n in contains['wrestler_name'].head(5).tolist():
                print(f"      - {n}")
        else:
            print(f"   ❌ No matches found for last name '{last_name}'")


🔍 IDENTIFYING MISSING BIG TEN WRESTLERS

📊 125lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 133lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 141lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 149lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 157lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 165lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 174lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 184lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 197lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📊 285lbs - Checking 14 wrestlers...
   ✅ Found: 14/14 wrestlers

📋 SUMMARY: 0 total missing wrestlers across all weight classes

💡 NAME CORRECTION SUGGESTIONS:
------------------------------------------------------------

📝 UPDATED WRESTLER LISTS (with only found wrestlers)

125lbs (14 wrestlers):
   - Ayden Smith
   - Nicolar Rivera
   - Jore Volk
   - Spencer Moore
   

In [88]:
# ============================================
# FUNCTION TO FIND MISSING IVY LEAGUE WRESTLERS
# ============================================

def find_missing_ivy_wrestlers(wrestler_lists, weight_classes, inference_db):
    """
    Identify which Ivy League wrestlers from each weight class are missing from the inference database.

    Parameters:
    - wrestler_lists: list of lists containing wrestler names by weight class
    - weight_classes: list of weight class numbers
    - inference_db: the inference pipeline dataframe

    Returns:
    - missing_dict: dictionary with weight class as key and list of missing wrestlers as value
    - found_dict: dictionary with weight class as key and list of found wrestlers as value
    """

    print("\n" + "="*100)
    print("🔍 IDENTIFYING MISSING IVY LEAGUE WRESTLERS")
    print("="*100)

    missing_dict = {}
    found_dict = {}
    all_missing = []

    for weight_idx, (weight, wrestler_list) in enumerate(zip(weight_classes, wrestler_lists)):
        print(f"\n📊 {weight}lbs - Checking {len(wrestler_list)} wrestlers...")

        # Find which wrestlers are in the database
        found = inference_db[inference_db['wrestler_name'].isin(wrestler_list)]['wrestler_name'].tolist()
        missing = [w for w in wrestler_list if w not in found]

        found_dict[weight] = found
        missing_dict[weight] = missing
        all_missing.extend(missing)

        # Print results for this weight class
        print(f"   ✅ Found: {len(found)}/{len(wrestler_list)} wrestlers")
        if found:
            print(f"   📋 Found wrestlers:")
            for w in found:
                # Get their data
                wrestler_data = inference_db[inference_db['wrestler_name'] == w].iloc[0]
                print(f"      - {w} (Rank: {wrestler_data['rank']}, Team: {wrestler_data['team_name']})")

        if missing:
            print(f"   ❌ Missing ({len(missing)}):")
            for w in missing:
                # Try to find similar names for suggestions
                similar = inference_db[inference_db['wrestler_name'].str.contains(w.split()[-1] if ' ' in w else w, case=False, na=False)]['wrestler_name'].tolist()
                print(f"      - {w}")
                if similar:
                    print(f"        → Similar names in DB: {similar[:3]}")

    print("\n" + "="*100)
    print(f"📋 SUMMARY: {len(all_missing)} total missing wrestlers across all weight classes")
    print("="*100)

    return missing_dict, found_dict


# ============================================
# FUNCTION TO SUGGEST CORRECTIONS FOR IVY LEAGUE
# ============================================

def suggest_ivy_name_corrections(missing_dict, inference_db):
    """
    Suggest possible corrections for missing Ivy League wrestler names.
    """
    print("\n💡 IVY LEAGUE NAME CORRECTION SUGGESTIONS:")
    print("-" * 60)

    all_suggestions = {}

    for weight, missing_list in missing_dict.items():
        if not missing_list:
            continue

        print(f"\n{weight}lbs:")
        weight_suggestions = {}

        for missing_name in missing_list:
            # Split name to get last name
            parts = missing_name.split()
            if len(parts) > 1:
                last_name = parts[-1]
                # Find similar last names in database
                similar = inference_db[inference_db['wrestler_name'].str.contains(last_name, case=False, na=False)]
                if len(similar) > 0:
                    suggestions = similar['wrestler_name'].tolist()
                    weight_suggestions[missing_name] = suggestions
                    print(f"   '{missing_name}' → maybe: {suggestions}")
                else:
                    print(f"   '{missing_name}' → no close matches found")
            else:
                print(f"   '{missing_name}' → single name, can't parse last name")

        all_suggestions[weight] = weight_suggestions

    return all_suggestions


# ============================================
# IVY LEAGUE WRESTLER LISTS
# ============================================

ivy_wrestlers_125 = ["Davis Motyka", "Greg Diakomihalis", "Logan Brzozowski", "Sulayman Bah", "Chase Yasutake", "Marc-Anthony McGowan"]

ivy_wrestlers_133 = ["Evan Mougalian", "Ethan Rivera", "Coleman Nogle", "Evin Gursoy", "Douglas Shipers", "Tyler Ferrera"]

ivy_wrestlers_141 = ["Vince Cornella", "Matthew Martino", "Lorenzo Frezza", "Khimari Manns", "Dante Frinzi", "CJ Composto"]

ivy_wrestlers_149 = ["Jaxon Joy", "Richard Fedalen", "Jack Crook", "Eligh Rivera", "Austin McBurney", "Cross Wasilewski"]

ivy_wrestlers_157 = ["Meyer Shapiro", "Ethan Mojena", "Kai Owen", "Jimmy Harrington", "Rocco Camillaci", "Jude Swisher"]

ivy_wrestlers_165 = ["Cesar Alvan", "Joseph Cangro", "Maximus Norman", "Sean Seefeldt", "Benny Rogers", "Ty Whalen"]

ivy_wrestlers_174 = ["Simon Ruiz", "Haden Bottiglieri", "Caden Bellis", "Holden Garcia", "Drew Clearie", "Nick Fine"]

ivy_wrestlers_184 = ["Joe Curtis", "Xavier Giles", "Matthew Walsh", "Liam Carlin", "Connor O'Donnell", "Louis Cerchio"]

ivy_wrestlers_197 = ["Andrew Reall", "Aiden Hanning", "Hudson Skove", "Martin Cosgrove", "Conor McCloskey", "Jack Wehmeyer"]

ivy_wrestlers_285 = ["Vincent Mueller", "Sebastian Garibaldi", "Ashton Davis", "John Pardo", "Daniel Bittner", "Alex Semenenko"]

# ============================================
# COMBINE INTO MASTER LIST
# ============================================

ivy_all_wrestler_list = [
    ivy_wrestlers_125, ivy_wrestlers_133, ivy_wrestlers_141, ivy_wrestlers_149,
    ivy_wrestlers_157, ivy_wrestlers_165, ivy_wrestlers_174, ivy_wrestlers_184,
    ivy_wrestlers_197, ivy_wrestlers_285
]

ivy_weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

# ============================================
# RUN THE ANALYSIS FOR IVY LEAGUE
# ============================================

print("\n" + "="*100)
print("🏆 IVY LEAGUE WRESTLER VERIFICATION")
print("="*100)

# Find missing wrestlers
ivy_missing_dict, ivy_found_dict = find_missing_ivy_wrestlers(
    ivy_all_wrestler_list, ivy_weights, inference_pipeline_db
)

# Get correction suggestions
ivy_suggestions = suggest_ivy_name_corrections(ivy_missing_dict, inference_pipeline_db)

# ============================================
# CREATE UPDATED IVY LEAGUE LISTS
# ============================================

print("\n" + "="*100)
print("📝 UPDATED IVY LEAGUE WRESTLER LISTS (with only found wrestlers)")
print("="*100)

ivy_updated_lists = []
for weight in ivy_weights:
    updated_list = ivy_found_dict[weight]
    ivy_updated_lists.append(updated_list)
    print(f"\n{weight}lbs ({len(updated_list)} wrestlers):")
    for w in updated_list:
        wrestler_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == w].iloc[0]
        print(f"   - {w} (Rank: {wrestler_data['rank']}, Team: {wrestler_data['team_name']})")

# ============================================
# SUMMARY TABLE
# ============================================

print("\n" + "="*100)
print("📊 IVY LEAGUE WRESTLER SUMMARY TABLE")
print("="*100)

summary_data = []
for weight in ivy_weights:
    original_count = len(ivy_all_wrestler_list[ivy_weights.index(weight)])
    found_count = len(ivy_found_dict[weight])
    missing_count = original_count - found_count

    summary_data.append({
        'Weight': f"{weight}lbs",
        'Original': original_count,
        'Found': found_count,
        'Missing': missing_count,
        'Status': '✅ COMPLETE' if missing_count == 0 else f'⚠️ MISSING {missing_count}'
    })

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

# ============================================
# CHECK SPECIFIC IVY LEAGUE NAMES
# ============================================

print("\n" + "="*100)
print("🔎 CHECKING SPECIFIC IVY LEAGUE NAME PATTERNS")
print("="*100)

# Common Ivy League naming issues to check
ivy_problematic = [
    "Marc-Anthony McGowan",  # Has hyphen
    "CJ Composto",           # Initials
    "AShton Davis",          # Capitalization issue
    "Haden Bottiglieri",     # Uncommon name
    "Louis Cerchio",         # Uncommon name
    "Khimari Manns",         # Uncommon name
]

for name in ivy_problematic:
    print(f"\nChecking '{name}':")

    # Exact match
    exact = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == name]
    if len(exact) > 0:
        data = exact.iloc[0]
        print(f"   ✅ Found: {data['wrestler_name']} (Rank: {data['rank']}, Team: {data['team_name']})")
        continue

    # Try contains (case insensitive)
    parts = name.split()
    if len(parts) > 1:
        last_name = parts[-1].replace('-', ' ')  # Handle hyphenated names
        contains = inference_pipeline_db[inference_pipeline_db['wrestler_name'].str.contains(last_name, case=False, na=False)]
        if len(contains) > 0:
            print(f"   🔍 Found by last name '{last_name}':")
            for _, row in contains.head(3).iterrows():
                print(f"      - {row['wrestler_name']} (Rank: {row['rank']}, Team: {row['team_name']})")
        else:
            # Try first name
            first_name = parts[0]
            contains_first = inference_pipeline_db[inference_pipeline_db['wrestler_name'].str.contains(first_name, case=False, na=False)]
            if len(contains_first) > 0:
                print(f"   🔍 Found by first name '{first_name}':")
                for _, row in contains_first.head(3).iterrows():
                    print(f"      - {row['wrestler_name']} (Rank: {row['rank']}, Team: {row['team_name']})")
            else:
                print(f"   ❌ No matches found")
    else:
        print(f"   ❌ Single name, can't parse")

print("\n" + "="*100)
print("✅ IVY LEAGUE WRESTLER VERIFICATION COMPLETE")
print("="*100)


🏆 IVY LEAGUE WRESTLER VERIFICATION

🔍 IDENTIFYING MISSING IVY LEAGUE WRESTLERS

📊 125lbs - Checking 6 wrestlers...
   ✅ Found: 6/6 wrestlers
   📋 Found wrestlers:
      - Sulayman Bah (Rank: 23, Team: Columbia)
      - Logan Brzozowski (Rank: 45, Team: Harvard)
      - Chase Yasutake (Rank: 230, Team: Brown)
      - Marc-Anthony McGowan (Rank: 13, Team: Princeton)
      - Greg Diakomihalis (Rank: 50, Team: Cornell)
      - Davis Motyka (Rank: 37, Team: Pennsylvania)

📊 133lbs - Checking 6 wrestlers...
   ✅ Found: 6/6 wrestlers
   📋 Found wrestlers:
      - Douglas Shipers (Rank: 254, Team: Brown)
      - Evin Gursoy (Rank: 68, Team: Columbia)
      - Coleman Nogle (Rank: 72, Team: Harvard)
      - Tyler Ferrera (Rank: 18, Team: Cornell)
      - Evan Mougalian (Rank: 16, Team: Pennsylvania)
      - Ethan Rivera (Rank: 45, Team: Princeton)

📊 141lbs - Checking 6 wrestlers...
   ✅ Found: 6/6 wrestlers
   📋 Found wrestlers:
      - Vince Cornella (Rank: 6, Team: Cornell)
      - Matthew M